# XJTU discharge forecasting and controller benchmark

Measured-reference SOH and delivered-energy forecasts cover 10, 20, and 30 operating cycles. The 30-cycle endpoints are also scored after measurements arrive at 10 and 20 cycles. All learned forecasts exclude the evaluated battery from fitting and model selection.

The discharge controller study compares the original Lookahead (deterministic MPC), forecast-based MPC, risk-aware MPC, Discrete SAC, PPO, FQI, and simple baselines. Every policy is evaluated in the same frozen Notebook 20 simulator.

Run from the project directory or one of its descendants. Inputs are the XJTU data directory or ZIP and the Notebook 19–21 report folders or ZIPs. The configuration cell allows explicit paths. The notebook uses one CPU thread.

The forecast refinement compares bounded SOH and Wh corrections against each previous recommendation. The saved controller benchmark can be reused for forecast-only runs.

In [1]:
import os
for thread_var in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
                   "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[thread_var] = "1"

import io
import json
import pickle
import sys
from itertools import product
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics.pairwise import euclidean_distances
from torch import nn
from torch.distributions import Categorical
from threadpoolctl import threadpool_limits

torch.set_num_threads(1)

cwd = Path.cwd()
project = next((p for p in (cwd, *cwd.parents) if (p / "README_LifeGuard_RL.md").is_file()), None)
if project is None:
    raise FileNotFoundError("Open this notebook from within the LifeGuard-RL project.")
local_dependencies = project / ".venv" / "notebook20-deps"
if local_dependencies.is_dir() and str(local_dependencies) not in sys.path:
    sys.path.insert(0, str(local_dependencies))
import gymnasium as gym

SOURCE_OVERRIDE = None
PILOT_OVERRIDE = None
OUTPUT_ROOT = None
DATASET_PATH = None
NOTEBOOK_21_EXPORT = None
RERUN_CONTROLLER_BENCHMARK = False  # Run original comparison if a compatible cache is unavailable.
INPUT_NAME = "lifeguard_xjtu_discharge_soh_results"
OUTPUT_NAME = "lifeguard_xjtu_discharge_forecast_sac_results"
PILOT_NAME = "lifeguard_xjtu_discharge_ppo_results"
source = Path(SOURCE_OVERRIDE).expanduser() if SOURCE_OVERRIDE else project / "reports" / INPUT_NAME
if SOURCE_OVERRIDE is None and not source.is_dir():
    source = source.with_suffix(".zip")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else project / "reports") / OUTPUT_NAME

SEED = 42
PPO_SEEDS = (42, 73, 101)
BOOTSTRAP_DRAWS = 2000
HORIZON = 3
DISTURBANCES = 8
MIN_SOH = 80.0
SOH_RADIUS = 6.0
AGE_RADIUS = 70.0
MIN_CELLS = 2
ENERGY_FRACTION = .90
HIT_TOLERANCE = .05
ENERGY_ALPHA = 10
PRESETS = (0, 1, 2)
PROFILE_COLUMNS = ["first_current_mean_a", "initial_current_square_mean_a2",
                   "initial_duration_mean_min", "initial_current_square_minutes"]
POLICIES = {"fixed": "Fixed middle template", "rule": "SOH rule",
            "lookahead": "Constrained lookahead", "ppo": "PPO"}



pilot = Path(PILOT_OVERRIDE).expanduser() if PILOT_OVERRIDE else project / "reports" / PILOT_NAME
if PILOT_OVERRIDE is None and not pilot.is_dir():
    pilot = pilot.with_suffix(".zip")
FORECAST_SEEDS = (42, 73, 101)
RIDGE_ALPHAS = (1, 10, 100)
TRANSFORMER_EPOCHS = 70
TRAIN_BRANCHES = 8
PLANNING_PATHS = 32
PLANNING_ENERGY_PROBABILITY = .95
ENERGY_RATE_TOLERANCE = .01
TAIL_FRACTION = .10
TOLERANCE = 1e-9

POLICIES = {
    "ppo": "PPO", "fqi": "Fitted Q iteration", "lookahead": "Current lookahead",
    "scenario": "Scenario-based lookahead",
}
FORECAST_LABELS = {
    "recursive": "Notebook 19 recursive Ridge", "direct": "Direct Ridge",
    "history": "Direct Ridge with added history", "transformer": "Compact Transformer",
    "residual": "Residual Transformer",
}


def artifact(filename, directory):
    if directory.is_dir():
        return (directory / filename).read_bytes()
    with ZipFile(directory) as archive:
        files = [member for member in archive.namelist() if PurePosixPath(member).name == filename]
        if len(files) != 1:
            raise ValueError(f"Expected one {filename} in {directory}.")
        return archive.read(files[0])


def read_table(name, directory=source, **options):
    return pd.read_csv(io.BytesIO(artifact(name + ".csv", directory)), **options)

benchmark = Path(NOTEBOOK_21_EXPORT).expanduser() if NOTEBOOK_21_EXPORT else project / "reports/lifeguard_xjtu_forecast_fqi_results"
if NOTEBOOK_21_EXPORT is None and not benchmark.is_dir():
    benchmark = benchmark.with_suffix(".zip")
raw_source = Path(DATASET_PATH).expanduser() if DATASET_PATH else project / "Dataset/XJTU"
if DATASET_PATH is None and not raw_source.is_dir():
    raw_source = raw_source.with_suffix(".zip")

SEEDS = (42, 73, 101)
SAC_UPDATES = 1500
SAC_BATCH = 64
TAU = .005
BLOCK = 10
GRU_EPOCHS = 48
HORIZONS = (1, 2, 3)
FORECAST_WEIGHTS = np.array([.30, .40, .30])
PLANNING_RISK = 0.95
STRESS_SCALE = 1.25
ADVANCEMENT = .10
FEATURE_VOLTAGES = (4.0, 3.8, 3.6, 3.4, 3.2, 3.0)
POLICIES = {
    "fixed": "Fixed middle template", "rule": "SOH rule", "lookahead": "Original Lookahead",
    "mpc": "Forecast MPC", "risk": "Risk-aware MPC", "ppo": "PPO", "fqi": "Fitted Q iteration",
    "sac": "Discrete SAC",
}

In [2]:
def add_histories(intervals):
    data = intervals.loc[intervals.status.eq("valid")].copy().reset_index(drop=True)
    end_at = {(row.cell_id, row.end_cycle): row for row in data.itertuples(index=False)}
    histories = []
    for row in data.itertuples(index=False):
        previous = end_at.get((row.cell_id, row.start_cycle))
        earlier = end_at.get((row.cell_id, previous.start_cycle)) if previous is not None else None
        histories.append((previous.soh_before if previous is not None else np.nan,
                          earlier.soh_before if earlier is not None else np.nan))
    data[["previous_soh", "earlier_soh"]] = histories
    return data


def chain_indices(data, horizon):
    by_start = {(row.cell_id, row.start_cycle): row.Index for row in data.itertuples()}
    routes = []
    for row in data.itertuples():
        path = [row.Index]
        for _ in range(1, horizon):
            forward = by_start.get((row.cell_id, data.at[path[-1], "end_cycle"]))
            if forward is None:
                break
            path.append(forward)
        if len(path) == horizon:
            routes.append(path)
    return np.asarray(routes, dtype=int).reshape(-1, horizon)


gate = read_table("ppo_gate")
if len(gate) != 1 or not bool(gate.iloc[0].advance_to_ppo):
    raise ValueError("The Notebook 19 report must pass the PPO pilot gate.")
intervals = read_table("reference_intervals")
data = add_histories(intervals.loc[intervals.cohort.eq("RW")])
known = read_table("held_out_predictions")
known = known.loc[known.cohort.eq("RW")]
choices = read_table("model_selection")
choices = choices.loc[choices.cohort.eq("RW") & choices.model.eq("action") & choices.selected]
alpha_by_cell = choices.set_index("cell_id", verify_integrity=True).alpha.to_dict()
cells = sorted(data.cell_id.unique())
cell_ids = data.cell_id.to_numpy()
profiles = data[PROFILE_COLUMNS].to_numpy()
initial = data[["soh_before", "previous_soh", "earlier_soh"]].to_numpy()
ages = data.operating_age.to_numpy()
truth = np.column_stack((data.soh_after - data.soh_before, data.realized_wh))
chains = {h: chain_indices(data, h) for h in (1, HORIZON)}
assert set(alpha_by_cell) == set(cells) and np.isfinite(profiles).all() and np.isfinite(truth).all()


def design(states, age, commands):
    soh, previous, earlier = states.T
    has_prev = np.isfinite(previous)
    recent = np.where(has_prev, soh - previous, 0)
    drift = np.where(np.isfinite(earlier), (soh - earlier) / 2, recent)
    return np.column_stack((soh, age, recent, drift, has_prev, commands))


def fit(train, health_alpha):
    x = design(initial[train], ages[train], profiles[train])
    subset = data.loc[train]
    weights = 1 / subset.groupby("cell_id").start_cycle.transform("size").to_numpy(dtype=float)
    weights /= weights.mean()
    usable = np.ptp(x, axis=0) > 0
    matrix = x[:, usable]
    center = np.average(matrix, axis=0, weights=weights)
    scale = np.sqrt(np.average((matrix - center)**2, axis=0, weights=weights))
    standardized = (matrix - center) / scale
    slopes = np.zeros((x.shape[1], 2))
    intercept = np.empty(2)
    for j, alpha in enumerate((health_alpha, ENERGY_ALPHA)):
        ridge = Ridge(alpha=alpha).fit(standardized, truth[train, j], sample_weight=weights)
        slopes[usable, j] = ridge.coef_ / scale
        intercept[j] = ridge.intercept_ - center @ slopes[usable, j]
    return slopes, intercept


def transition(model, state, age, commands):
    coefficients, intercept = model
    return design(state, age, commands) @ coefficients + intercept


def calibrate(train, held_out):
    errors = np.full_like(truth, np.nan)
    for inner in cells:
        if inner == held_out:
            continue
        score = cell_ids == inner
        fit_rows = train & ~score
        model = fit(fit_rows, health_alpha=10)
        errors[score] = truth[score] - transition(model, initial[score], ages[score], profiles[score])
    paths = chains[HORIZON]
    paths = paths[train[paths[:, 0]]]
    noise = errors[paths]
    if not np.isfinite(noise).all():
        raise ValueError("Incomplete training-only calibration windows.")
    return noise


def measure_forecasts(model, held_out, horizons):
    records = []
    for h, all_paths in horizons.items():
        paths = all_paths[cell_ids[all_paths[:, 0]] == held_out]
        state = initial[paths[:, 0]].copy()
        wh = np.zeros(len(paths))
        for step in paths.T:
            estimate = transition(model, state, ages[step], profiles[step])
            wh += estimate[:, 1]
            state = np.column_stack((state[:, 0] + estimate[:, 0], state[:, :2]))
        for path, predicted, delivered in zip(paths, state[:, 0], wh):
            start = data.iloc[path[0]]
            end = data.iloc[path[-1]]
            records.append((held_out, int(start.start_cycle), int(end.end_cycle), h,
                            predicted, end.soh_after, delivered, float(truth[path, 1].sum())))
    return records


with threadpool_limits(limits=1):
    models, disturbances, forecast_records = {}, {}, []
    for cell in cells:
        train = cell_ids != cell
        models[cell] = fit(train, alpha_by_cell[cell])
        disturbances[cell] = calibrate(train, cell)
        forecast_records.extend(measure_forecasts(models[cell], cell, chains))

forecast_checks = pd.DataFrame(forecast_records, columns=["cell_id", "start_cycle", "end_cycle", "horizon_tests",
    "soh_prediction", "soh_measured", "wh_prediction", "wh_measured"])
joined = forecast_checks.merge(known[["cell_id", "start_cycle", "end_cycle", "horizon_tests", "action"]],
    on=["cell_id", "start_cycle", "end_cycle", "horizon_tests"], validate="one_to_one")
assert len(joined) == len(known) == len(forecast_checks)
np.testing.assert_allclose(joined.soh_prediction, joined.action, atol=1e-9, rtol=0)
print("Reproduced all Notebook 19 held-out action-aware SOH forecasts.")

def forecast_features(table, paths):
    lookup = {(row.cell_id, row.end_cycle): row for row in table.itertuples(index=False)}
    planned = table[PROFILE_COLUMNS].to_numpy()[paths]
    anchor_indices = paths[:, 0]
    states = table[["soh_before", "previous_soh", "earlier_soh"]].to_numpy()[anchor_indices]
    base = design(states, table.operating_age.to_numpy()[anchor_indices], planned[:, 0])[:, :5]
    tokens = np.zeros((len(paths), 8, 8), dtype=np.float32)
    mask = np.zeros((len(paths), 8), dtype=bool)
    history = []
    for i, index in enumerate(anchor_indices):
        row = table.iloc[index]
        prior = lookup.get((row.cell_id, row.start_cycle))
        older = lookup.get((row.cell_id, prior.start_cycle)) if prior is not None else None
        points = [(row.soh_before, prior)]
        previous = prior
        while previous is not None and len(points) < 5:
            latest = lookup.get((row.cell_id, previous.start_cycle))
            points.append((previous.soh_before, latest))
            previous = latest
        readings = [value for value, _ in points]
        trend = (readings[0] - readings[-1]) / (len(readings) - 1) if len(readings) > 1 else 0
        curvature = readings[0] - 2 * readings[1] + readings[2] if len(readings) > 2 else 0
        history.append((
            trend, curvature,
            prior.realized_wh if prior is not None else np.nan,
            prior.realized_discharge_minutes if prior is not None else np.nan,
            prior.realized_wh - older.realized_wh if older is not None else 0,
            prior.realized_discharge_minutes - older.realized_discharge_minutes if older is not None else 0,
        ))
        mask[i, :5 - len(points)] = True
        for offset, (value, past) in enumerate(points):
            tokens[i, 4 - offset, :4] = (
                (value - row.soh_before) / 5,
                (past.realized_wh - 65) / 10 if past is not None else 0,
                (past.realized_discharge_minutes - 220) / 70 if past is not None else 0,
                float(past is not None),
            )
    tokens[:, 5:, 4:] = (planned - np.array([5, 25, 4, 100])) / np.array([2, 15, 1, 50])
    direct = np.column_stack((base, planned.mean(axis=1)))
    extended = np.column_stack((direct, history, planned[:, -1] - planned[:, 0]))
    return direct, extended, tokens, mask


def verify_forecast_timing(table, paths):
    route = paths[-1:]
    changed = table.copy()
    changed.loc[route.ravel(), ["realized_wh", "realized_discharge_minutes", "soh_after"]] = -999
    for original, alternate in zip(forecast_features(table, route), forecast_features(changed, route)):
        np.testing.assert_array_equal(original, alternate)
    varied = table.copy()
    varied.loc[route[0, -1], PROFILE_COLUMNS[0]] += 1
    assert not np.array_equal(forecast_features(table, route)[1], forecast_features(varied, route)[1], equal_nan=True)


forecast_paths = chains[HORIZON]
verify_forecast_timing(data, forecast_paths)
direct_x, extensive_x, seq_x, seq_mask = forecast_features(data, forecast_paths)
first = forecast_paths[:, 0]
last = forecast_paths[:, -1]
forecast_cells = cell_ids[first]
origin_soh = initial[first, 0]
future_soh = data.soh_after.to_numpy()[last]
target_change = future_soh - origin_soh

def training_weights(train):
    sample = pd.Series(forecast_cells[train])
    weights = 1 / sample.groupby(sample).transform("size").to_numpy(dtype=float)
    return weights / weights.mean()


def scale_training(matrix, train):
    source = matrix[train]
    medians = np.nanmedian(source, axis=0)
    missing = np.isnan(matrix)
    filled = np.column_stack((np.where(missing, medians, matrix), missing[:, missing[train].any(axis=0)]))
    weights = training_weights(train)
    center = np.average(filled[train], axis=0, weights=weights)
    scale = np.sqrt(np.average((filled[train] - center)**2, axis=0, weights=weights))
    scale = np.maximum(scale, 1e-8)
    return np.clip((filled - center) / scale, -8, 8)


def ridge_prediction(matrix, train, alpha):
    scaled = scale_training(matrix, train)
    fit = Ridge(alpha=alpha).fit(scaled[train], target_change[train], sample_weight=training_weights(train))
    return fit.predict(scaled)


def choose_alpha(matrix, pool):
    inner_groups = np.unique(forecast_cells[pool])
    estimates = {}
    for alpha in RIDGE_ALPHAS:
        errors = []
        for cell in inner_groups:
            test = forecast_cells == cell
            forecast = ridge_prediction(matrix, pool & ~test, alpha)
            errors.append(np.abs(target_change[test] - forecast[test]).mean())
        estimates[alpha] = np.mean(errors)
    return min(RIDGE_ALPHAS, key=lambda alpha: (estimates[alpha], -alpha))

Reproduced all Notebook 19 held-out action-aware SOH forecasts.


In [3]:
class XJTUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"Unexpected global in XJTU data: {module}.{name}")

def curve_features(cycle):
    """Trapezoidal discharge Ah and Wh at first downward voltage crossings."""
    current, voltage, time = (np.asarray(cycle[key], dtype=float) for key in
                               ("current_in_A", "voltage_in_V", "time_in_s"))
    active = current < -.25
    dt = np.diff(time) * (active[:-1] & active[1:])
    ah = np.r_[0., np.cumsum(dt * (-current[1:] - current[:-1]) / 7200)]
    power = -voltage * current
    wh = np.r_[0., np.cumsum(dt * (power[1:] + power[:-1]) / 7200)]
    values = []
    for threshold in FEATURE_VOLTAGES:
        matches = np.flatnonzero(active[:-1] & active[1:] & (voltage[:-1] >= threshold) & (voltage[1:] < threshold))
        if not len(matches):
            values.append(np.nan)
            continue
        ix = matches[0]
        fraction = (voltage[ix]-threshold) / (voltage[ix]-voltage[ix+1])
        values.append(ah[ix] + fraction*(ah[ix+1]-ah[ix]))
        if threshold == 3.0:
            wh_under_cutoff = wh[ix] + fraction*(wh[ix+1]-wh[ix])
    return [*values, wh_under_cutoff if np.isfinite(values[-1]) else np.nan]


def read_curves(path, wanted):
    def from_battery(stream, cell):
        battery = XJTUUnpickler(stream).load()
        assert battery["cell_id"] == cell and abs(battery["nominal_capacity_in_Ah"] - 2.) < 1e-8
        return [(cell, number, *curve_features(cycle)) for cycle in battery["cycle_data"]
                if (number := int(cycle["cycle_number"])) in wanted[cell] and cycle["attribute"] == "RPT"]

    records = []
    if path.is_dir():
        for cell in sorted(wanted):
            with (path / f"{cell}.pkl").open("rb") as stream:
                records.extend(from_battery(stream, cell))
    else:
        with ZipFile(path) as archive:
            names = {}
            for member in archive.namelist():
                entry = PurePosixPath(member)
                if entry.suffix == ".pkl" and entry.stem in wanted and not any(p.startswith(".") for p in entry.parts):
                    if entry.stem in names:
                        raise ValueError(f"Multiple XJTU files for {entry.stem}.")
                    names[entry.stem] = member
            for cell in sorted(wanted):
                with archive.open(names[cell]) as stream:
                    records.extend(from_battery(stream, cell))
    wanted_keys = {(cell, cycle) for cell, cycles in wanted.items() for cycle in cycles}
    assert len(records) == len(wanted_keys) and {(row[0], row[1]) for row in records} == wanted_keys
    return pd.DataFrame(records, columns=["cell_id", "cycle", *CURVE_FIELDS])


CURVE_FIELDS = [*(f"ah_at_{value:.1f}v" for value in FEATURE_VOLTAGES), "wh_at_3.0v"]
cycle_audit = read_table("cycle_audit")
rw_audit = cycle_audit.loc[cycle_audit.cohort.eq("RW")]
wanted = {cell: set(rows.cycle) for cell, rows in rw_audit.loc[rw_audit.status.eq("valid_rpt")].groupby("cell_id")}
curve_export = output / "reference_curve_features.csv"
if DATASET_PATH is None and curve_export.is_file():
    cached_curves = pd.read_csv(curve_export, float_precision="round_trip")
    wanted_keys = {(cell, cycle) for cell, cycles in wanted.items() for cycle in cycles}
    found_keys = set(zip(cached_curves.cell_id, cached_curves.cycle))
    if wanted_keys != found_keys or len(cached_curves) != len(wanted_keys):
        raise ValueError("Reference curve cache does not match the archived XJTU audit.")
    rpt_curves = cached_curves[["cell_id", "cycle", *CURVE_FIELDS]]
    print("Reusing verified reference curve features.")
else:
    rpt_curves = read_curves(raw_source, wanted)
measured_cycles = rw_audit.loc[rw_audit.status.eq("valid_operating")].set_index(["cell_id", "cycle"], verify_integrity=True)
assert len(measured_cycles) == 1650 and len(rpt_curves) == 173
print(f"Loaded {len(rpt_curves)} reference curves and {len(measured_cycles)} measured operating cycles.")

Reusing verified reference curve features.
Loaded 173 reference curves and 1650 measured operating cycles.


## Forecast design

The forecast origin is the end of a completed reference test. Historical SOH and operating energy use only preceding completed intervals. The first current and duration of each future cycle are prescribed inputs. R3 contributes only to the optional shared SOH trend. Model selection uses nested leave-one-battery-out validation. Curve, transfer, sequence, and combined SOH/Wh approaches are reported against the same measured windows.

In [4]:
@dataclass
class ForecastInputs:
    health: np.ndarray
    past_wh: np.ndarray
    curve: np.ndarray
    age: np.ndarray
    commands: np.ndarray

    def take(self, selection):
        return ForecastInputs(self.health[selection], self.past_wh[selection], self.curve[selection],
                              self.age[selection], self.commands[selection])


@dataclass
class RecordedWindows:
    inputs: ForecastInputs
    indexes: np.ndarray
    cells: np.ndarray
    starts: np.ndarray
    ends: np.ndarray
    y_soh: np.ndarray
    y_cycle_wh: np.ndarray


def snapshot_tables(table, curves, diagnostics):
    prior_interval = {(row.cell_id, row.end_cycle): row for row in table.itertuples(index=False)}
    rates = curves.set_index(["cell_id", "cycle"])
    first = rates.reset_index().groupby("cell_id", sort=False).first()[CURVE_FIELDS]
    lookup = {cell: rows.set_index("cycle").diagnostic_soh_pct.sort_index()
              for cell, rows in diagnostics.groupby("cell_id")}
    health, energy, shape = [], [], []
    for row in table.itertuples(index=False):
        readings = lookup[row.cell_id].loc[:row.start_cycle].tail(5).to_numpy()[::-1]
        np.testing.assert_allclose(readings[0], row.soh_before, atol=1e-9)
        past = []
        stop = row.start_cycle
        for _ in range(3):
            past_row = prior_interval.get((row.cell_id, stop))
            if past_row is None:
                break
            assert past_row.end_cycle <= row.start_cycle
            past.append(past_row.realized_wh)
            stop = past_row.start_cycle
        current = rates.loc[(row.cell_id, row.start_cycle), CURVE_FIELDS].to_numpy(dtype=float)
        original = first.loc[row.cell_id].to_numpy(dtype=float)
        health.append(np.pad(readings, (0, 5-len(readings)), constant_values=np.nan))
        energy.append(np.pad(np.asarray(past, dtype=float), (0, 3-len(past)), constant_values=np.nan))
        shape.append(np.r_[current, current[:6]-original[:6]])
    return np.asarray(health), np.asarray(energy), np.asarray(shape)


def make_windows(paths):
    first_indices, last_indices = paths[:, 0], paths[:, -1]
    indexes = paths.ravel()
    starts, ends = data.start_cycle.to_numpy(), data.end_cycle.to_numpy()
    result = RecordedWindows(
        ForecastInputs(health_history[first_indices], wh_history[first_indices], curve_snapshot[first_indices],
                       ages[first_indices], commands[paths]),
        paths, cell_ids[first_indices], starts[first_indices], ends[last_indices],
        data.soh_after.to_numpy()[last_indices],
        cycle_wh[indexes].reshape(len(paths), -1),
    )
    expected = truth[paths, 1].sum(axis=1)
    np.testing.assert_allclose(result.y_cycle_wh.sum(axis=1), expected, atol=1e-9)
    return result


def known_profile(plans):
    amps, minutes = plans[..., 0], plans[..., 1]
    return np.stack((amps.mean(axis=-1), (amps**2).mean(axis=-1),
                     minutes.mean(axis=-1), (amps**2*minutes).mean(axis=-1)), axis=-1)


def verify_snapshots():
    case = chains[3][-1]
    altered = data.copy()
    altered.loc[case, ["realized_wh", "soh_after"]] = -999.
    a, b, c = snapshot_tables(altered, rpt_curves, rw_audit.loc[rw_audit.status.eq("valid_rpt")])
    start = case[0]
    for original, replacement in ((health_history, a), (wh_history, b), (curve_snapshot, c)):
        np.testing.assert_array_equal(original[start], replacement[start])
    future = windows[3].inputs.commands[-1:].copy()
    original = known_profile(future)
    future[0, -1, 0, 0] += 1
    assert not np.array_equal(known_profile(future), original)


keys = [(row.cell_id, cycle) for row in data.itertuples(index=False)
        for cycle in range(row.start_cycle+1, row.end_cycle)]
ordered_cycles = measured_cycles.loc[keys]
commands = ordered_cycles[["first_current_a", "verified_first_duration_s"]].to_numpy().reshape(-1, BLOCK, 2)
commands[..., 1] /= 60
cycle_wh = ordered_cycles.discharge_wh.to_numpy().reshape(-1, BLOCK)
np.testing.assert_allclose(known_profile(commands), profiles, atol=1e-8)
health_history, wh_history, curve_snapshot = snapshot_tables(
    data, rpt_curves, rw_audit.loc[rw_audit.status.eq("valid_rpt")])
chains[2] = chain_indices(data, 2)
windows = {h: make_windows(chains[h]) for h in HORIZONS}
assert [len(windows[h].cells) for h in HORIZONS] == [165, 157, 149]
verify_snapshots()
print("Verified every prescribed command profile, measured Wh total and causal origin snapshot.")


def history_terms(health):
    differences = health[:, :-1]-health[:, 1:]
    recent = np.where(np.isfinite(differences[:, 0]), differences[:, 0], 0.)
    previous = np.where(np.isfinite(differences[:, 1]), differences[:, 1], recent)
    known = np.isfinite(differences)
    weight = known*np.array([4., 3., 2., 1.])
    drift = (weight*np.nan_to_num(differences)).sum(axis=1)/np.maximum(weight.sum(axis=1), 1)
    peak = np.nanmax(health, axis=1)
    decline = np.maximum(0, peak-health[:, 0])
    return np.column_stack((recent, previous, drift, recent-previous, decline, known.sum(axis=1)))


def anchors(samples, h):
    health = samples.health[:, 0]
    recent, previous, drift, curvature, decline, count = history_terms(samples.health).T
    early = (samples.age <= 55) & (recent >= -.30) & (decline <= .4)
    typical = np.clip(.65*recent+.35*drift + .3*np.minimum(0, curvature), -4, 3)
    accelerating = np.clip(.8*recent+.2*previous, -4, 0)
    multiplier = np.clip((np.abs(recent)+.25)/(np.abs(previous)+.25), .94, 1.27)
    future = np.stack([np.where(early, .53**(j+1)*recent,
                                np.where(recent < -.3, accelerating*multiplier**(j+1), typical))
                       for j in range(h)], axis=1)
    population = -(np.maximum(0, 95-health)/14 + .14)
    future = np.where((count == 0)[:, None], population[:, None], future)
    measured = samples.past_wh
    weights = np.isfinite(measured)*np.array([5., 2., 1.])
    level = (np.nan_to_num(measured)*weights).sum(axis=1)/np.maximum(weights.sum(axis=1), 1)
    level = np.where(np.isfinite(measured[:, 0]), level, samples.curve[:, 6]*10)
    level = np.where(np.isfinite(level), level, 65*health/95)
    level = np.clip(level, 1, 90)
    drift_wh = np.where(np.isfinite(measured[:, 1]), measured[:, 0]-measured[:, 1], 0.)
    prior = np.clip(level[:, None] + np.clip(drift_wh, -3, 2)[:, None]*np.arange(1, h+1)*.45, 0, 90)
    return future, prior


def tabular(samples, h, use_curve=True):
    first = samples.health[:, 0]
    schedule = known_profile(samples.commands)
    distance = schedule[:, -1]-schedule[:, 0]
    wh = samples.past_wh
    terms = history_terms(samples.health)
    fields = [first, samples.age, np.maximum(90-first, 0), terms, wh, wh[:, 0]-wh[:, 1],
              schedule.mean(axis=1), distance]
    if use_curve:
        fields.append(samples.curve)
    return np.column_stack(fields)


def cycle_design(samples, h, coupled_change=None):
    raw = samples.commands.reshape(-1, h*BLOCK, 2)
    amps, minutes = raw[..., 0], raw[..., 1]
    stress = amps**2 * minutes
    n = len(amps)
    lead = np.tile(np.arange(h*BLOCK)[None, :], (n, 1))
    prefix = np.cumsum(stress, axis=1)/(lead+1)
    fields = (amps, minutes, amps*minutes, stress, prefix, lead//BLOCK, lead % BLOCK)
    shared = np.column_stack((samples.health[:, 0], samples.age, history_terms(samples.health),
                              samples.past_wh, samples.curve[:, [1, 3, 5, 6, 12]]))
    features = np.concatenate((np.stack(fields, axis=-1),
                               np.repeat(shared[:, None, :], h*BLOCK, axis=1)), axis=-1)
    _, anchor_wh = anchors(samples, h)
    prior = np.repeat(anchor_wh/BLOCK, BLOCK, axis=1)
    if coupled_change is not None:
        midpoint = np.cumsum(coupled_change, axis=1)-.5*coupled_change
        horizon_health = np.repeat(midpoint, BLOCK, axis=1)
        soh_now = samples.health[:, :1]
        prior = prior * np.clip((soh_now+horizon_health)/soh_now, .5, 1.3)
        features = np.concatenate((features, horizon_health[..., None]), axis=-1)
    return features, prior

Verified every prescribed command profile, measured Wh total and causal origin snapshot.


In [5]:
from sklearn.kernel_approximation import Nystroem


def weights_for(groups):
    groups = pd.Series(groups)
    weight = 1 / groups.groupby(groups).transform("size").to_numpy(float)
    return weight/weight.mean()


class TrainingScale:
    def __init__(self, matrix, weights):
        self.medians = np.array([np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.
                                 for col in matrix.T])
        self.missing = ~np.isfinite(matrix).all(axis=0)
        x = self.fill(matrix)
        self.center = np.average(x, axis=0, weights=weights)
        self.scale = np.sqrt(np.average((x-self.center)**2, axis=0, weights=weights))
        self.scale = np.maximum(self.scale, 1e-6)

    def fill(self, matrix):
        missing = ~np.isfinite(matrix)
        return np.column_stack((np.where(missing, self.medians, matrix), missing[:, self.missing]))

    def transform(self, matrix):
        return np.clip((self.fill(matrix)-self.center)/self.scale, -8, 8)


def fit_regressor(matrix, targets, weights, estimator):
    scale = TrainingScale(matrix, weights)
    estimator.fit(scale.transform(matrix), targets, sample_weight=weights)
    return scale, estimator


def regress(model, matrix):
    scale, estimator = model
    return estimator.predict(scale.transform(matrix))


def portable_history(health, age):
    return np.column_stack((health[:, 0], age, np.maximum(90-health[:, 0], 0), history_terms(health)))


def fit_transfer(h):
    rows = []
    r3 = intervals.loc[intervals.cohort.eq("R3") & intervals.status.eq("valid")]
    for cell, trace in r3.groupby("cell_id"):
        items = list(trace.sort_values("start_cycle").itertuples(index=False))
        for ix in range(len(items)-2*h+1):
            start = items[ix]
            target = items[ix+2*h-1]
            if target.end_cycle-start.start_cycle != 12*h:
                continue
            readings = np.full(5, np.nan)
            for j, step in enumerate(range(ix, max(-1, ix-10), -2)):
                if items[step].start_cycle == start.start_cycle-12*j:
                    readings[j] = items[step].soh_before
            rows.append((cell, readings, start.operating_age, target.soh_after-start.soh_before))
    groups = [r[0] for r in rows]
    assert len(set(groups)) == 8
    matrix = portable_history(np.stack([r[1] for r in rows]), np.asarray([r[2] for r in rows]))
    return fit_regressor(matrix, np.asarray([r[3] for r in rows]), weights_for(groups), Ridge(alpha=10))


TRANSFER = {h: fit_transfer(h) for h in HORIZONS}
SOH_MODELS = ("trend", "acceleration", "recursive", "phase_kernel", "analogs", "history_ridge", "curve_ridge", "huber", "kernel", "boost", "r3_transfer")
WH_MODELS = ("wh_anchor", "wh_direct", "wh_analogs", "wh_age", "wh_cycle_matched", "cycle_ridge", "cycle_kernel", "cycle_boost", "cycle_gru", "cycle_soh")


def peer_predictions(test, reference, group, labels):
    weights = weights_for(group)
    scaler = TrainingScale(reference, weights)
    matched = scaler.transform(reference)
    result = []
    for begin in range(0, len(test), 512):
        distance = euclidean_distances(scaler.transform(test[begin:begin+512]), matched)
        peers = []
        distances = []
        for cell in np.unique(group):
            positions = np.flatnonzero(group == cell)
            best = positions[distance[:, positions].argmin(axis=1)]
            distances.append(distance[np.arange(len(distance)), best])
            peers.append(labels[best])
        distances, peers = np.stack(distances, axis=1), np.stack(peers, axis=1)
        closest = np.argsort(distances, axis=1, kind="stable")[:, :3]
        weight = 1/(np.take_along_axis(distances, closest, axis=1)+.5)**2
        result.append(np.sum(np.take_along_axis(peers, closest, axis=1)*weight, axis=1)/weight.sum(axis=1))
    return np.concatenate(result)


def soh_recursive(model, samples):
    state = samples.health[:, :3].copy()
    profile = known_profile(samples.commands)
    change = []
    for step, action in enumerate(profile.transpose(1, 0, 2)):
        estimate = transition(model, state, samples.age+BLOCK*step, action)
        change.append(estimate[:, 0])
        state = np.column_stack((state[:, 0]+estimate[:, 0], state[:, :2]))
    return np.stack(change, axis=1)


class CycleGRU(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(width, 16), nn.Tanh())
        self.sequence = nn.GRU(16, 16, batch_first=True)
        self.head = nn.Linear(16, 1)

    def forward(self, values):
        latent, _ = self.sequence(self.embed(values))
        return self.head(latent).squeeze(-1)


def fit_cycle_gru(matrix, y, scale, weights, seeds):
    x = torch.as_tensor(scale.transform(matrix.reshape(-1, matrix.shape[-1])).reshape(*matrix.shape[:2], -1), dtype=torch.float32)
    truth = torch.as_tensor(y, dtype=torch.float32)
    weight = torch.as_tensor(weights.reshape(y.shape), dtype=torch.float32)
    nets = []
    for seed in seeds:
        torch.manual_seed(seed)
        net = CycleGRU(x.shape[-1])
        optimizer = torch.optim.AdamW(net.parameters(), lr=.006, weight_decay=.02)
        for _ in range(GRU_EPOCHS):
            optimizer.zero_grad()
            predicted = net(x)
            errors = nn.functional.smooth_l1_loss(predicted, truth, reduction="none", beta=.1)
            total_error = nn.functional.smooth_l1_loss(predicted.mean(1), truth.mean(1), reduction="none", beta=.1)
            loss = (errors*weight).mean()+.75*(total_error*weight.mean(1)).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1)
            optimizer.step()
        nets.append(net.eval())
    return nets


def boost(depth=2):
    return HistGradientBoostingRegressor(max_iter=60, learning_rate=.055, max_depth=depth, min_samples_leaf=10,
                                         l2_regularization=3, early_stopping=False, random_state=SEED)


@lru_cache(maxsize=None)
def fit_cycle_family(pool, omit=None):
    pack = windows[3]
    use = np.isin(pack.cells, pool)
    if omit is not None:
        use &= ~((pack.starts < omit[1]) & (pack.ends >= omit[0]))
    sample = pack.inputs.take(use)
    groups = pack.cells[use]
    target = pack.y_cycle_wh[use]
    history_model = fit(np.isin(cell_ids, pool), 1)
    coupled = np.empty((len(groups), 3))
    for battery in np.unique(groups):
        mask = groups == battery
        model = fit(np.isin(cell_ids, pool) & (cell_ids != battery), 1)
        coupled[mask] = soh_recursive(model, sample.take(mask))
    a, prior = cycle_design(sample, 3)
    b, coupled_prior = cycle_design(sample, 3, coupled)
    cycle_keys = [(cell, start+j+j//BLOCK+1) for cell, start in zip(groups, pack.starts[use])
                  for j in range(3*BLOCK)]
    occurrences = pd.Series(cycle_keys).value_counts()
    repeated_weights = np.asarray([1/occurrences[key] for key in cycle_keys]).reshape(-1, 3*BLOCK)
    group_weight = pd.DataFrame({"cell": groups, "weight": repeated_weights.sum(axis=1)})
    repeated_weights /= group_weight.groupby("cell").weight.transform("sum").to_numpy()[:, None]
    weights = repeated_weights.ravel()
    weights /= weights.mean()
    matrix = a.reshape(-1, a.shape[-1])
    scale = TrainingScale(matrix, weights)
    x = scale.transform(matrix)
    residual = target-prior
    basis = Nystroem(kernel="rbf", gamma=.035, n_components=min(48, len(x)), random_state=SEED).fit(x)
    return dict(scale=scale, ridge=Ridge(alpha=18).fit(x, residual.ravel(), sample_weight=weights),
                basis=basis, kernel=Ridge(alpha=8).fit(basis.transform(x), residual.ravel(), sample_weight=weights),
                boost=boost(3).fit(x, residual.ravel(), sample_weight=weights),
                gru=fit_cycle_gru(a, residual, scale, weights, SEEDS if len(pool) == 7 and omit is None else (SEED,)),
                paired=fit_regressor(b.reshape(-1, b.shape[-1]), (target-coupled_prior).ravel(), weights, Ridge(alpha=18)),
                soh_model=history_model)


def predict_cycle_family(model, inputs, h, names):
    base, prior = cycle_design(inputs, h)
    matrix = model["scale"].transform(base.reshape(-1, base.shape[-1]))
    result = {}
    for name in names:
        if name == "cycle_ridge":
            offsets = model["ridge"].predict(matrix).reshape(prior.shape)
        elif name == "cycle_kernel":
            offsets = model["kernel"].predict(model["basis"].transform(matrix)).reshape(prior.shape)
        elif name == "cycle_boost":
            offsets = model["boost"].predict(matrix).reshape(prior.shape)
        elif name == "cycle_gru":
            with torch.inference_mode():
                x = torch.as_tensor(matrix.reshape(*base.shape[:2], -1), dtype=torch.float32)
                offsets = np.mean([net(x).numpy() for net in model["gru"]], axis=0)
        else:
            paired, adjusted = cycle_design(inputs, h, soh_recursive(model["soh_model"], inputs))
            offsets = regress(model["paired"], paired.reshape(-1, paired.shape[-1])).reshape(prior.shape)
            result[name] = np.maximum(0., adjusted+offsets).sum(axis=1)
            continue
        result[name] = np.maximum(0., prior+offsets).sum(axis=1)
    return result


@lru_cache(maxsize=None)
def fitted_heads(pool, h, omit=None):
    pack = windows[h]
    use = np.isin(pack.cells, pool)
    if omit is not None:
        use &= ~((pack.starts < omit[1]) & (pack.ends >= omit[0]))
    samples = pack.inputs.take(use)
    group = pack.cells[use]
    weight = weights_for(group)
    y = pack.y_soh[use]-samples.health[:, 0]
    wh = pack.y_cycle_wh[use].sum(axis=1)
    local, wh_anchor = anchors(samples, h)
    local = local.sum(axis=1)
    wh_anchor = wh_anchor.sum(axis=1)
    context, wave = tabular(samples, h, False), tabular(samples, h)
    transfer = regress(TRANSFER[h], portable_history(samples.health, samples.age))
    return dict(
        pool=pool,
        history=fit_regressor(context, y, weight, Ridge(alpha=10)),
        curve=fit_regressor(wave, y-local, weight, Ridge(alpha=18)),
        huber=fit_regressor(wave, y-local, weight, HuberRegressor(alpha=.8, max_iter=1200)),
        kernel=fit_regressor(wave, y-local, weight, KernelRidge(alpha=.65, kernel="rbf", gamma=.025)),
        boost=fit_regressor(wave, y-local, weight, boost()),
        transfer=fit_regressor(wave, y-transfer, weight, Ridge(alpha=25)),
        wh=fit_regressor(wave, wh-wh_anchor, weight, Ridge(alpha=12)),
        peers=(wave, group, y-local, wh-wh_anchor, pack.starts[use], pack.ends[use]),
        cycles=fit_cycle_family(pool, omit),
    )


def acceleration_forecast(samples, h, trend):
    history = samples.health
    first, second, third = (history[:, j+1]-history[:, j] for j in range(3))
    supported = (first >= .55) & (second >= .3) & (third >= .25) & (history[:, 0] < 92)
    current = np.maximum(first, .2)
    momentum = .7*np.log(current/np.maximum(second, .2))+.3*np.log(np.maximum(second, .2)/np.maximum(third, .2))
    pace = np.exp(np.clip(momentum, np.log(.85), np.log(1.34)))
    accumulated = -current * sum(pace**j for j in range(1, h+1))
    return np.where(supported, accumulated, trend)


def refined_recursion(pool, sample, h):
    one_step = fitted_heads(pool, 1)
    health, past_wh, curve = sample.health.copy(), sample.past_wh.copy(), sample.curve.copy()
    first = health[:, 0].copy()
    for offset in range(h):
        current = ForecastInputs(health, past_wh, curve, sample.age+offset*BLOCK,
                                 sample.commands[:, offset:offset+1])
        next_soh, next_wh = candidate_predictions(one_step, current, 1,
                                 ("kernel", "curve_ridge", "acceleration"), ("wh_direct",))
        next_health = .35*next_soh["kernel"]+.40*next_soh["curve_ridge"]+.25*next_soh["acceleration"]
        ratio = np.clip(next_health/np.maximum(health[:, 0], 1), .88, 1.08)
        curve[:, :7] *= ratio[:, None]
        curve[:, 7:] += curve[:, :6]*(1-1/ratio[:, None])
        past_wh = np.column_stack((next_wh["wh_direct"], past_wh[:, :2]))
        health = np.column_stack((next_health, health[:, :4]))
    return health[:, 0]-first


@lru_cache(maxsize=None)
def cycle_matches(pool):
    one = windows[1]
    use = np.isin(one.cells, pool)
    observed = one.inputs.take(use)
    _, anchor = anchors(observed, 1)
    residual = one.y_cycle_wh[use]-anchor/BLOCK
    age = observed.age[:, None]+np.arange(BLOCK)
    return age.ravel(), observed.commands.reshape(-1, 2), residual.ravel()


def matched_cycle_wh(pool, samples, h):
    ref_age, ref_schedule, ref_residual = cycle_matches(pool)
    _, wh = anchors(samples, h)
    prior = np.repeat(wh/BLOCK, BLOCK, axis=1)
    age = (samples.age[:, None]+np.arange(h*BLOCK)).ravel()
    action = samples.commands.reshape(-1, 2)
    applied = np.empty(len(age))
    for ix in range(0, len(age), 256):
        stop = ix+256
        age_dist = np.abs(age[ix:stop, None]-ref_age)
        amp_dist = np.abs(action[ix:stop, :1]-ref_schedule[:, 0])
        min_dist = np.abs(action[ix:stop, 1:2]-ref_schedule[:, 1])
        weight = np.exp(-age_dist/2.-amp_dist/1.4-min_dist/2.5)
        applied[ix:stop] = (weight @ ref_residual)/(weight.sum(axis=1)+1.5)
    return np.maximum(0., prior+applied.reshape(prior.shape)).sum(axis=1)


def candidate_predictions(model, inputs, h, soh_names=SOH_MODELS, wh_names=WH_MODELS):
    local, anchor_wh = anchors(inputs, h)
    trend = local.sum(axis=1)
    anchor_wh = anchor_wh.sum(axis=1)
    wave = tabular(inputs, h)
    health = inputs.health[:, 0]
    refs, group, soh_resid, wh_resid, _, _ = model["peers"]
    soh, wh = {}, {}
    for name in soh_names:
        if name == "trend":
            prediction = trend
        elif name == "acceleration":
            prediction = acceleration_forecast(inputs, h, trend)
        elif name == "phase_kernel":
            fitted = regress(model["kernel"], wave)
            prediction = acceleration_forecast(inputs, h, trend)+np.where(health < 90, .25, 1.)*fitted
        elif name == "recursive":
            prediction = refined_recursion(model["pool"], inputs, h)
        elif name == "analogs":
            prediction = trend+peer_predictions(wave, refs, group, soh_resid)
        elif name == "history_ridge":
            prediction = regress(model["history"], tabular(inputs, h, False))
        elif name == "r3_transfer":
            prediction = regress(TRANSFER[h], portable_history(inputs.health, inputs.age))+regress(model["transfer"], wave)
        else:
            key = "curve" if name == "curve_ridge" else name
            prediction = trend+regress(model[key], wave)
        soh[name] = np.clip(health+prediction, 0, 105)
    for name in wh_names:
        if name == "wh_anchor":
            wh[name] = anchor_wh
        elif name == "wh_direct":
            wh[name] = anchor_wh+regress(model["wh"], wave)
        elif name == "wh_analogs":
            wh[name] = anchor_wh+peer_predictions(wave, refs, group, wh_resid)
        elif name == "wh_cycle_matched":
            wh[name] = matched_cycle_wh(model["pool"], inputs, h)
        elif name == "wh_age":
            reference_age = refs[:, 1]
            dist = np.abs(inputs.age[:, None]-reference_age)
            nearby = np.exp(-dist/8)
            correction = (nearby @ wh_resid)/(nearby.sum(axis=1)+1)
            wh[name] = anchor_wh+correction
    cycle_names = [name for name in wh_names if name.startswith("cycle_")]
    if cycle_names:
        wh.update(predict_cycle_family(model["cycles"], inputs, h, cycle_names))
    return soh, wh

In [6]:
TARGETS = ("soh", "wh")
LIBRARY = (SOH_MODELS, WH_MODELS)


def upper_tail(values):
    values = np.asarray(values)
    return np.sort(values)[-int(np.ceil(len(values)*TAIL_FRACTION)):].mean()


def errors_by_group(actual, predicted, groups):
    raw = np.abs(actual-predicted)
    by_cell = pd.Series(raw).groupby(groups).mean()
    return np.array([by_cell.mean(), upper_tail(raw), raw.max()])


def truth_matrix(pack):
    return np.column_stack((pack.y_soh, pack.y_cycle_wh.sum(axis=1)))


@lru_cache(maxsize=None)
def old_predictions(pool, h):
    pack = windows[h]
    model = fit(np.isin(cell_ids, pool), 1)
    # Kept identical to Notebook 20, including recursive SOH states during Wh rollouts.
    readings = initial[pack.indexes[:, 0]].copy()
    total = np.zeros(len(readings))
    for step in pack.indexes.T:
        estimates = transition(model, readings, ages[step], profiles[step])
        total += estimates[:, 1]
        readings = np.column_stack((readings[:, 0]+estimates[:, 0], readings[:, :2]))
    health = readings[:, 0]
    if h == 3:
        train = np.isin(forecast_cells, pool)
        alpha = choose_alpha(extensive_x, train)
        health = origin_soh+ridge_prediction(extensive_x, train, alpha)
    return np.column_stack((health, total))


def conservative_choice(actual, predictions, group):
    baseline = errors_by_group(actual, predictions["baseline"], group)
    metrics = {name: errors_by_group(actual, predicted, group) for name, predicted in predictions.items()}
    scores = {name: float(FORECAST_WEIGHTS @ (value/np.maximum(baseline, 1e-9))) for name, value in metrics.items()}
    options = sorted((name for name in predictions if name != "baseline"), key=lambda name: (scores[name], name))[:4]
    blends = {}
    for i, left in enumerate(options):
        for right in options[i+1:]:
            for weight in (.25, .5, .75):
                name = (left, right, weight)
                pred = (1-weight)*predictions[left]+weight*predictions[right]
                blends[name] = errors_by_group(actual, pred, group)
    assessed = {**metrics, **blends}
    eligible = [name for name, values in assessed.items() if (values <= baseline+1e-9).all()]
    selected = min(eligible, key=lambda name: (
        FORECAST_WEIGHTS @ (assessed[name]/np.maximum(baseline, 1e-9)), str(name)))
    return selected, assessed


def use_choice(spec, options):
    if isinstance(spec, tuple):
        left, right, weight = spec
        return (1-weight)*options[left]+weight*options[right]
    return options[spec]


def choice_label(name):
    if isinstance(name, tuple):
        return f"{name[0]} + {name[1]} ({name[2]:g} second)"
    return name


def required_names(choice):
    return choice[:2] if isinstance(choice, tuple) else (choice,)


def peer_age_check(model, sample, h, first, last):
    refs, groups, soh_resid, wh_resid, ref_first, ref_last = model["peers"]
    health, energy = anchors(sample, h)
    wave = tabular(sample, h)
    ref_age = refs[:, 1]
    result = []
    for ix, (start, stop) in enumerate(zip(first, last)):
        use = (ref_last <= start) | (ref_first >= stop)
        if not use.any():
            result.append((np.nan, np.nan, np.nan, 0))
            continue
        soh = peer_predictions(wave[ix:ix+1], refs[use], groups[use], soh_resid[use])[0]
        wh = peer_predictions(wave[ix:ix+1], refs[use], groups[use], wh_resid[use])[0]
        similarity = np.exp(-np.abs(sample.age[ix]-ref_age[use])/8)
        outside_stage = (similarity @ wh_resid[use])/(similarity.sum()+1)
        result.append((sample.health[ix, 0]+health[ix].sum()+soh, energy[ix].sum()+wh,
                       energy[ix].sum()+outside_stage, use.sum()))
    return result


def archived_checks(held_out, h, test, base):
    pack = windows[h]
    keys = ["cell_id", "start_cycle", "end_cycle"]
    index = pd.MultiIndex.from_arrays((pack.cells[test], pack.starts[test], pack.ends[test]), names=keys)
    if h in (1, 3):
        past = ppo_forecasts.loc[ppo_forecasts.horizon_tests.eq(h)].set_index(keys).loc[index]
        np.testing.assert_allclose(base[test, 1], past.wh_prediction, atol=1e-9, rtol=0)
        np.testing.assert_allclose(pack.y_soh[test], past.soh_measured, atol=1e-9, rtol=0)
        np.testing.assert_allclose(pack.y_cycle_wh[test].sum(axis=1), past.wh_measured, atol=1e-9, rtol=0)
        source = past.soh_prediction if h == 1 else archived_history.loc[index, "history"]
        np.testing.assert_allclose(base[test, 0], source, atol=1e-9, rtol=0)
    assert set(pack.cells[test]) == {held_out}


@lru_cache(maxsize=None)
def validate_candidates(pool, excluded, h):
    pack = windows[h]
    mask = pack.cells == excluded
    model = fitted_heads(pool, h)
    estimates = candidate_predictions(model, pack.inputs.take(mask), h)
    reference = old_predictions(pool, h)[mask]
    for index in range(2):
        estimates[index]["baseline"] = reference[:, index]
    return estimates


def record_predictions():
    flat, selection_rows, age_rows = [], [], []
    all_models = {}
    for held_out in cells:
        pool = tuple(cell for cell in cells if cell != held_out)
        all_models[held_out] = {}
        for h in HORIZONS:
            pack = windows[h]
            y = truth_matrix(pack)
            train = np.isin(pack.cells, pool)
            test = ~train
            score = [{name: np.full(len(y), np.nan) for name in (*names, "baseline")} for names in LIBRARY]
            for inner in pool:
                inner_pool = tuple(cell for cell in pool if cell != inner)
                inner_mask = pack.cells == inner
                guesses = validate_candidates(inner_pool, inner, h)
                for j, target in enumerate(TARGETS):
                    for name, values in guesses[j].items():
                        score[j][name][inner_mask] = values
            model = fitted_heads(pool, h)
            estimates = validate_candidates(pool, held_out, h)
            old = old_predictions(pool, h)
            archived_checks(held_out, h, test, old)
            specs, inner_stream = {}, []
            for j, target in enumerate(TARGETS):
                choice, metrics = conservative_choice(y[train, j],
                    {name: values[train] for name, values in score[j].items()}, pack.cells[train])
                specs[target] = choice
                for name, values in metrics.items():
                    selection_rows.append((held_out, h*BLOCK, target, choice_label(name), *values, name == choice))
                estimates[j]["selected"] = use_choice(choice, estimates[j])
                inner_stream.append(use_choice(choice, score[j]))
                for name, output in estimates[j].items():
                    for row, guess in zip(np.flatnonzero(test), output):
                        flat.append((held_out, int(pack.starts[row]), int(pack.ends[row]), h*BLOCK, target,
                                     name, y[row, j], float(guess), float(pack.inputs.health[row, 0])))
            all_models[held_out][h] = dict(pool=pool, fitted=model, choices=specs,
                                         inner=np.column_stack(inner_stream), train=train)
            if h == 3:
                for row, values in zip(np.flatnonzero(test), peer_age_check(
                        model, pack.inputs.take(test), h, pack.starts[test], pack.ends[test])):
                    age_rows.append((held_out, int(pack.starts[row]), int(pack.ends[row]), *y[row], *values))
        print(f"Forecast fold {held_out}: SOH {choice_label(specs['soh'])}; Wh {choice_label(specs['wh'])}", flush=True)
    return (pd.DataFrame(flat, columns=["cell_id", "start_cycle", "end_cycle", "horizon_cycles", "target", "model",
                                            "measured", "prediction", "start_soh"]),
            pd.DataFrame(selection_rows, columns=["held_out", "horizon_cycles", "target", "model",
                                                 "inner_mae", "inner_worst_10", "inner_max", "selected"]),
            pd.DataFrame(age_rows, columns=["cell_id", "start_cycle", "end_cycle", "measured_soh", "measured_wh",
                                          "soh_analogs", "wh_analogs", "wh_age", "eligible_training_windows"]),
            all_models)


def paired_interval(values):
    random = np.random.default_rng(SEED)
    indexes = random.integers(len(values), size=(BOOTSTRAP_DRAWS, len(values)))
    return np.quantile(values[indexes].mean(axis=1), [0.025, 0.975])


def by_forecast_group(table, groups):
    rows = []
    for keys, segment in table.groupby(groups, sort=True):
        baseline = segment.loc[segment.model.eq("baseline")].set_index(["cell_id", "start_cycle"], verify_integrity=True)
        reference = errors_by_group(baseline.measured.to_numpy(), baseline.prediction.to_numpy(),
                                    baseline.index.get_level_values(0))
        for name, group in segment.groupby("model"):
            group = group.set_index(["cell_id", "start_cycle"], verify_integrity=True).loc[baseline.index]
            np.testing.assert_allclose(group.measured, baseline.measured, atol=1e-9, rtol=0)
            measured, prediction, labels = group.measured.to_numpy(), group.prediction.to_numpy(), group.index.get_level_values(0)
            values = errors_by_group(measured, prediction, labels)
            delta = np.abs(measured-prediction)-np.abs(measured-baseline.prediction.to_numpy())
            ci = paired_interval(pd.Series(delta).groupby(labels).mean().to_numpy())
            worst = group.iloc[int(np.argmax(np.abs(measured-prediction)))].name
            improvement = 100*(1-values/reference)
            rows.append((*keys, name, len(prediction), *values, *improvement, *ci, *worst))
    return pd.DataFrame(rows, columns=[*groups, "model", "windows", "mae", "worst_10", "max_error", "mae_improvement_pct",
                                       "tail_improvement_pct", "max_improvement_pct", "mae_ci95_low", "mae_ci95_high",
                                       "worst_cell", "worst_start_cycle"])


def update_study():
    parent = windows[3]
    route_lookup = [{tuple(indexes): i for i, indexes in enumerate(windows[h].indexes)} for h in HORIZONS]
    indexed = forecast_predictions.set_index(["cell_id", "start_cycle", "horizon_cycles", "target", "model"]).prediction
    rows, tuning = [], []
    actual = truth_matrix(parent)
    for cell in cells:
        test = parent.cells == cell
        train = ~test
        for elapsed in (1, 2):
            h = 3-elapsed
            counts = [route_lookup[0][(path[elapsed-1],)] for path in parent.indexes]
            children = [route_lookup[h-1][tuple(path[elapsed:])] for path in parent.indexes]
            earlier = windows[1]
            end = windows[h]
            partial = np.column_stack((np.zeros(len(parent.cells)), truth[parent.indexes[:, :elapsed], 1].sum(axis=1)))
            # The latest completed ten-cycle error is available when we refresh the forecast.
            prev_inner = forecast_models[cell][1]["inner"][counts]
            next_inner = forecast_models[cell][h]["inner"][children]
            innovation = truth_matrix(earlier)[counts]-prev_inner
            for j, target in enumerate(TARGETS):
                nested = {}
                for coefficient in (0., .25, .5, 1.):
                    offset = coefficient*np.clip(innovation[:, j], -2., 2.)*np.sqrt(h)
                    nested[coefficient] = errors_by_group(actual[train, j],
                        (partial[:, j]+next_inner[:, j]+offset)[train], parent.cells[train])
                standard = np.maximum(nested[0.], 1e-9)
                eligible = [c for c, err in nested.items() if (err <= standard+1e-9).all()]
                chosen = min(eligible, key=lambda c: (FORECAST_WEIGHTS @ (nested[c]/standard), c))
                for coefficient, values in nested.items():
                    tuning.append((cell, elapsed*BLOCK, target, coefficient, *values, coefficient == chosen))
                for index in np.flatnonzero(test):
                    prior = counts[index]
                    upto = indexed.loc[(cell, int(earlier.starts[prior]), BLOCK, target, "selected")]
                    adjustment = chosen*np.clip(truth_matrix(earlier)[prior, j]-upto, -2., 2.)*np.sqrt(h)
                    when = int(end.starts[children[index]])
                    base = indexed.loc[(cell, when, h*BLOCK, target, "baseline")]
                    fresh = indexed.loc[(cell, when, h*BLOCK, target, "selected")]
                    values = [base, fresh, fresh+adjustment]
                    for method, guess in zip(("baseline", "fresh", "bias_corrected"), values):
                        rows.append((cell, int(parent.starts[index]), int(parent.ends[index]), elapsed*BLOCK,
                                     target, method, actual[index, j], partial[index, j]+guess,
                                     float(parent.inputs.health[index, 0])))
    return (pd.DataFrame(rows, columns=["cell_id", "start_cycle", "end_cycle", "elapsed_cycles", "target",
                                        "model", "measured", "prediction", "start_soh"]),
            pd.DataFrame(tuning, columns=["held_out", "elapsed_cycles", "target", "correction",
                                          "inner_mae", "inner_worst_10", "inner_max", "selected"]))


ppo_forecasts = read_table("recorded_forecasts", pilot)
archived_history = read_table("forecast_predictions", benchmark).set_index(["cell_id", "start_cycle", "end_cycle"])
with threadpool_limits(limits=1):
    forecast_predictions, forecast_selection, peer_sensitivity, forecast_models = record_predictions()
    rolling_predictions, rolling_selection = update_study()
forecast_metrics = by_forecast_group(forecast_predictions, ["horizon_cycles", "target"])
rolling_metrics = by_forecast_group(rolling_predictions, ["elapsed_cycles", "target"])
print(forecast_metrics.loc[forecast_metrics.model.isin(["baseline", "selected"]),
                           ["horizon_cycles", "target", "model", "mae", "worst_10", "max_error"]].round(3).to_string(index=False))

Forecast fold XJTU_RW_battery-1: SOH recursive + kernel (0.25 second); Wh wh_direct + cycle_soh (0.5 second)
Forecast fold XJTU_RW_battery-2: SOH baseline; Wh cycle_soh + wh_direct (0.5 second)
Forecast fold XJTU_RW_battery-3: SOH recursive + history_ridge (0.25 second); Wh cycle_gru + cycle_soh (0.5 second)
Forecast fold XJTU_RW_battery-4: SOH history_ridge + kernel (0.5 second); Wh wh_direct + cycle_soh (0.5 second)
Forecast fold XJTU_RW_battery-5: SOH kernel + phase_kernel (0.25 second); Wh cycle_soh + wh_direct (0.25 second)
Forecast fold XJTU_RW_battery-6: SOH recursive + boost (0.25 second); Wh cycle_soh + wh_direct (0.5 second)
Forecast fold XJTU_RW_battery-7: SOH recursive + boost (0.25 second); Wh wh_direct + cycle_soh (0.5 second)
Forecast fold XJTU_RW_battery-8: SOH recursive + boost (0.25 second); Wh wh_direct + cycle_soh (0.5 second)
 horizon_cycles target    model   mae  worst_10  max_error
             10    soh baseline 0.196     0.492      0.757
             10    soh 

In [7]:
# These are the previously recommended Notebook 23 forecasts. Keep the
# benchmark fixed while choosing a correction inside each battery fold.
FIXED_INCUMBENT = {
    1: ("selected", "selected"),
    2: ("baseline", "selected"),
    3: ("baseline", "selected"),
}
ROLLING_INCUMBENT = {1: ("baseline", "bias_corrected"), 2: ("bias_corrected", "bias_corrected")}
REFINEMENT_ALTERNATIVES = {"soh": ("kernel", "recursive"), "wh": ("wh_direct", "cycle_soh", "wh_analogs")}
BLEND_WEIGHTS = (.25, .5, .75)
SOH_CAPS = (.5, 1., 1.5)
WH_CAPS_PER_BLOCK = (.25, .5, 1.)
EPSILON = 1e-9


@dataclass
class RefinementView:
    mode: str
    cycles: int
    horizon: int
    parent: RecordedWindows
    at_origin: np.ndarray
    prior: np.ndarray | None
    measured_credit: np.ndarray
    wh_bias_inputs: np.ndarray

    @property
    def key(self):
        return self.mode, self.cycles


def bias_inputs(inputs):
    profile = known_profile(inputs.commands)
    prior = inputs.past_wh
    return np.column_stack((inputs.health[:, 0], inputs.age, prior[:, :2], prior[:, 0]-prior[:, 1],
                            inputs.curve[:, 6], profile.mean(axis=1)[:, [0, 2, 3]]))


def make_refinement_views():
    for h, pack in windows.items():
        zeros = np.zeros((len(pack.cells), 2))
        yield RefinementView("fixed", h*BLOCK, h, pack, np.arange(len(pack.cells)),
                             None, zeros, bias_inputs(pack.inputs))
    parent = windows[3]
    positions = {h: {tuple(path): j for j, path in enumerate(pack.indexes)} for h, pack in windows.items()}
    for elapsed in (1, 2):
        h = 3-elapsed
        child = np.array([positions[h][tuple(path[elapsed:])] for path in parent.indexes])
        prior = np.array([positions[1][(path[elapsed-1],)] for path in parent.indexes])
        credit = np.zeros((len(parent.cells), 2))
        credit[:, 1] = truth[parent.indexes[:, :elapsed], 1].sum(axis=1)
        latest = windows[h].inputs.take(child)
        yield RefinementView("rolling", elapsed*BLOCK, h, parent, child, prior, credit, bias_inputs(latest))


@lru_cache(maxsize=None)
def refinement_catalogs(pool, h, held_out=None):
    """Return full-length catalogs populated on the requested cells only."""
    pack = windows[h]
    options = tuple({name: np.full(len(pack.cells), np.nan) for name in (*names, "baseline")} for names in LIBRARY)
    scored_cells = (held_out,) if held_out is not None else pool
    for inner in scored_cells:
        mask = pack.cells == inner
        fitted_pool = tuple(cell for cell in pool if cell != inner)
        assert inner not in fitted_pool
        inner_predictions = validate_candidates(fitted_pool, inner, h)
        for target, library in zip(options, inner_predictions):
            for name in target:
                target[name][mask] = library[name]
    return options


@lru_cache(maxsize=None)
def v1_selections(pool, h):
    if len(pool) == len(cells)-1:
        outer = next(cell for cell in cells if cell not in pool)
        saved = forecast_models[outer][h]["choices"]
        return tuple(saved[target] for target in TARGETS)
    view = windows[h]
    use = np.isin(view.cells, pool)
    catalogs = refinement_catalogs(pool, h)
    observed = truth_matrix(view)
    return tuple(conservative_choice(observed[use, j], {name: values[use] for name, values in bank.items()},
                                     view.cells[use])[0] for j, bank in enumerate(catalogs))


def v1_selected(pool, h, held_out=None):
    return np.column_stack([use_choice(choice, bank) for choice, bank in
                            zip(v1_selections(pool, h), refinement_catalogs(pool, h, held_out))])


def rolling_state(view, pool, held_out=None):
    if view.prior is None:
        raise ValueError("A rolling update requires a previously completed block.")
    fresh = v1_selected(pool, view.horizon, held_out)[view.at_origin]
    prior_error = truth_matrix(windows[1])[view.prior]-v1_selected(pool, 1, held_out)[view.prior]
    innovation = np.clip(prior_error, -2., 2.)*np.sqrt(view.horizon)
    return view.measured_credit+fresh, innovation


@lru_cache(maxsize=None)
def v1_rolling_coefficients(pool, cycles):
    view = REFINEMENT_VIEWS["rolling", cycles]
    fresh, innovation = rolling_state(view, pool)
    use = np.isin(view.parent.cells, pool)
    actual = truth_matrix(view.parent)
    results = []
    for j in range(2):
        errors = {gain: errors_by_group(actual[use, j], (fresh[:, j]+gain*innovation[:, j])[use],
                                        view.parent.cells[use]) for gain in (0., .25, .5, 1.)}
        eligible = [gain for gain, values in errors.items() if (values <= errors[0.]+EPSILON).all()]
        results.append(min(eligible, key=lambda gain: (FORECAST_WEIGHTS @
                           (errors[gain]/np.maximum(errors[0.], EPSILON)), gain)))
    return np.asarray(results)


def revision_inputs(view, pool, held_out=None):
    h = view.horizon
    banks = refinement_catalogs(pool, h, held_out)
    incumbents = FIXED_INCUMBENT[h] if view.mode == "fixed" else ROLLING_INCUMBENT[view.cycles//BLOCK]
    if view.mode == "fixed":
        adjusted_v1 = v1_selected(pool, h, held_out)
    else:
        fresh, innovation = rolling_state(view, pool, held_out)
        adjusted_v1 = fresh+innovation*v1_rolling_coefficients(pool, view.cycles)
    roles = []
    for j, (target, bank, incumbent) in enumerate(zip(TARGETS, banks, incumbents)):
        original = view.measured_credit[:, j]+bank["baseline"][view.at_origin]
        current = original if incumbent == "baseline" else adjusted_v1[:, j]
        alternatives = {name: view.measured_credit[:, j]+bank[name][view.at_origin]
                        for name in REFINEMENT_ALTERNATIVES[target]}
        roles.append((original, current, alternatives))
    return roles


def fit_wh_bias(view, pool):
    _, incumbent, _ = revision_inputs(view, pool)[1]
    train = np.isin(view.parent.cells, pool)
    residual = truth_matrix(view.parent)[train, 1]-incumbent[train]
    return fit_regressor(view.wh_bias_inputs[train], residual, weights_for(view.parent.cells[train]), Ridge(alpha=30))


def correction_library(view, pool, held_out):
    h = view.horizon
    roles = revision_inputs(view, pool, held_out)
    bias = regress(fit_wh_bias(view, pool), view.wh_bias_inputs)
    result = []
    for j, (original, current, alternatives) in enumerate(roles):
        target = TARGETS[j]
        caps = SOH_CAPS if target == "soh" else tuple(h*cap for cap in WH_CAPS_PER_BLOCK)
        anchor = view.measured_credit[:, j]
        adjustments = {name: values-current for name, values in alternatives.items()}
        if target == "wh":
            adjustments["wh_bias"] = bias
        options = {"incumbent": current}
        for name, correction in adjustments.items():
            for weight in BLEND_WEIGHTS:
                for cap in (*caps, np.inf):
                    tag = "unbounded" if np.isinf(cap) else f"cap={cap:g}"
                    values = current+np.clip(weight*correction, -cap, cap)
                    label = f"{name} weight={weight:g} {tag}"
                    options[label] = np.maximum(anchor, values) if target == "wh" else np.clip(values, 0, 105)
        result.append((original, options))
    return result


def pick_revision(target, incumbent, alternatives, group):
    scores = {name: errors_by_group(target, values, group) for name, values in alternatives.items()}
    reference = scores[incumbent]
    eligible = [name for name, values in scores.items() if (values <= reference+EPSILON).all()]
    choice = min(eligible, key=lambda name: (FORECAST_WEIGHTS @ (scores[name]/np.maximum(reference, EPSILON)),
                                           name != incumbent, name))
    return choice, scores


def verify_refinement_incumbent(view, predictions):
    frame = forecast_predictions if view.mode == "fixed" else rolling_predictions
    time_col = "horizon_cycles" if view.mode == "fixed" else "elapsed_cycles"
    modes = FIXED_INCUMBENT[view.horizon] if view.mode == "fixed" else ROLLING_INCUMBENT[view.cycles//BLOCK]
    keys = ["cell_id", "start_cycle", "end_cycle"]
    for j, mode in enumerate(modes):
        expected = frame.loc[frame[time_col].eq(view.cycles) & frame.target.eq(TARGETS[j]) &
                             frame.model.eq(mode)].set_index(keys, verify_integrity=True)["prediction"]
        actual = predictions.loc[predictions.target.eq(TARGETS[j])].set_index(keys, verify_integrity=True)
        pd.testing.assert_index_equal(actual.sort_index().index, expected.sort_index().index)
        np.testing.assert_allclose(actual.incumbent, expected.loc[actual.index], rtol=0, atol=1e-9)


def evaluate_refinement():
    predictions, logs = [], []
    for view in REFINEMENT_VIEWS.values():
        rows = []
        parent = view.parent
        actual = truth_matrix(parent)
        for held_out in cells:
            train = parent.cells != held_out
            test = ~train
            pool = tuple(cell for cell in cells if cell != held_out)
            inner = [{}, {}]
            for excluded in pool:
                valid = parent.cells == excluded
                fit_pool = tuple(cell for cell in pool if cell != excluded)
                for j, (_, catalog) in enumerate(correction_library(view, fit_pool, excluded)):
                    for name, estimates in catalog.items():
                        inner[j].setdefault(name, np.full(len(parent.cells), np.nan))[valid] = estimates[valid]
            for j, (target, (original, catalog)) in enumerate(zip(TARGETS, correction_library(view, pool, held_out))):
                fit_rows = {name: values[train] for name, values in inner[j].items()}
                assert all(np.isfinite(values).all() for values in fit_rows.values())
                choice, assessed = pick_revision(actual[train, j], "incumbent", fit_rows, parent.cells[train])
                for name, score in assessed.items():
                    logs.append((*view.key, held_out, target, name, *score, name == choice))
                benchmark = "kernel" if target == "soh" else "wh_direct"
                equal = catalog[f"{benchmark} weight=0.5 unbounded"]
                for k in np.flatnonzero(test):
                    current = catalog["incumbent"][k]
                    rows.append((*view.key, target, held_out, int(parent.starts[k]), int(parent.ends[k]),
                                 float(parent.inputs.health[k, 0]), float(windows[view.horizon].inputs.health[view.at_origin[k], 0]),
                                 actual[k, j], original[k], current, equal[k], catalog[choice][k], choice))
        frame = pd.DataFrame(rows, columns=["mode", "cycles", "target", "cell_id", "start_cycle", "end_cycle",
            "start_soh", "forecast_origin_soh", "measured", "original", "incumbent", "equal_blend", "proposed", "rule"])
        verify_refinement_incumbent(view, frame)
        predictions.append(frame)
    return (pd.concat(predictions, ignore_index=True),
            pd.DataFrame(logs, columns=["mode", "cycles", "held_out", "target", "rule",
                                            "inner_mae", "inner_worst_10", "inner_max", "selected"]))


def summarize_refinement(table):
    summaries = []
    table["recommended"] = table.incumbent
    for (mode, cycles, target), rows in table.groupby(["mode", "cycles", "target"]):
        actual, group = rows.measured.to_numpy(), rows.cell_id.to_numpy()
        metrics = {name: errors_by_group(actual, rows[name].to_numpy(), group)
                   for name in ("original", "incumbent", "equal_blend", "proposed")}
        difference = metrics["proposed"]-metrics["incumbent"]
        advances = (difference <= EPSILON).all() and (difference < -EPSILON).any()
        if advances:
            table.loc[rows.index, "recommended"] = rows.proposed
        metrics["recommended"] = metrics["proposed"] if advances else metrics["incumbent"]
        for name, values in metrics.items():
            prediction = rows.proposed if name == "recommended" and advances else rows.incumbent if name == "recommended" else rows[name]
            error = (rows.measured-prediction).abs()
            deltas = error-(rows.measured-rows.incumbent).abs()
            interval = paired_interval(deltas.groupby(rows.cell_id).mean().to_numpy())
            worst = rows.loc[error.idxmax()]
            reductions = 100*(1-values/np.maximum(metrics["incumbent"], EPSILON))
            summaries.append((mode, cycles, target, name, len(rows), *values, *reductions, *interval,
                              worst.cell_id, int(worst.start_cycle), "revision" if advances else "incumbent"))
    return pd.DataFrame(summaries, columns=["mode", "cycles", "target", "role", "windows", "mae", "worst_10",
        "max_error", "mae_reduction_pct", "tail_reduction_pct", "max_reduction_pct", "mae_ci95_low", "mae_ci95_high",
        "worst_cell", "worst_start_cycle", "recommendation"])


def refinement_slices(table):
    summary, difficult = [], []
    for (mode, cycles, target), family in table.groupby(["mode", "cycles", "target"]):
        severe = family.loc[family.forecast_origin_soh.lt(90)]
        for role in ("incumbent", "proposed", "recommended"):
            if len(severe):
                scores = errors_by_group(severe.measured.to_numpy(), severe[role].to_numpy(), severe.cell_id.to_numpy())
                summary.append((mode, cycles, target, role, len(severe), *scores))
        focus = [(family.measured-family[role]).abs().idxmax() for role in ("incumbent", "proposed")]
        if mode == "fixed" and cycles == 30:
            for cell, cycle in (("XJTU_RW_battery-3", 307), ("XJTU_RW_battery-5", 241), ("XJTU_RW_battery-5", 230)):
                focus.extend(family.loc[family.cell_id.eq(cell) & family.start_cycle.eq(cycle)].index)
        for _, row in table.loc[list(dict.fromkeys(focus))].iterrows():
            errors = [abs(row.measured-row[role]) for role in ("original", "incumbent", "proposed", "recommended")]
            difficult.append((*[row[key] for key in ("mode", "cycles", "target", "cell_id", "start_cycle", "end_cycle")],
                              *errors))
    return (
        pd.DataFrame(summary, columns=["mode", "cycles", "target", "role", "windows", "mae", "worst_10", "max_error"]),
        pd.DataFrame(difficult, columns=["mode", "cycles", "target", "cell_id", "start_cycle", "end_cycle",
                                        "original_error", "incumbent_error", "proposed_error", "recommended_error"]),
    )


REFINEMENT_VIEWS = {view.key: view for view in make_refinement_views()}
with threadpool_limits(limits=1):
    refinement_predictions, refinement_selections = evaluate_refinement()
    refinement_metrics = summarize_refinement(refinement_predictions)
    refinement_health_bands, refinement_difficult_cases = refinement_slices(refinement_predictions)
print(refinement_metrics.loc[refinement_metrics.role.isin(["incumbent", "proposed"]),
    ["mode", "cycles", "target", "role", "mae", "worst_10", "max_error", "recommendation"]].round(3).to_string(index=False))


   mode  cycles target      role   mae  worst_10  max_error recommendation
  fixed      10    soh incumbent 0.123     0.345      0.528      incumbent
  fixed      10    soh  proposed 0.123     0.345      0.528      incumbent
  fixed      10     wh incumbent 0.259     0.762      1.203      incumbent
  fixed      10     wh  proposed 0.248     0.727      1.277      incumbent
  fixed      20    soh incumbent 0.409     1.167      1.586       revision
  fixed      20    soh  proposed 0.356     0.910      1.522       revision
  fixed      20     wh incumbent 0.485     1.440      2.156       revision
  fixed      20     wh  proposed 0.458     1.307      2.080       revision
  fixed      30    soh incumbent 0.598     2.035      2.603       revision
  fixed      30    soh  proposed 0.520     1.747      2.377       revision
  fixed      30     wh incumbent 0.909     2.291      3.378      incumbent
  fixed      30     wh  proposed 0.880     2.335      3.909      incumbent
rolling      10    soh in

## Controller evaluation

All policies use the original templates, starting states, energy budget, guard, and scoring dynamics. Forecast models only rank candidate sequences. Simulated observations can update SOH and accumulated Wh. Curve features remain those of the last available physical reference test. The base simulator and its evaluation disturbance paths are independent of the SAC and MPC training streams.

In [8]:
class ScheduleStudy:
    def __init__(self, held_out, model, noise):
        self.held_out = held_out
        self.model = model
        self.noise = noise
        self.train = np.flatnonzero(cell_ids != held_out)
        self.group_edges = np.quantile(profiles[self.train, -1], [1 / 3, 2 / 3])
        self.groups = np.searchsorted(self.group_edges, profiles[:, -1], side="right")
        self.train_cells = pd.factorize(cell_ids)[0]

    def nearby(self, soh, age, indexes):
        return (np.abs(initial[indexes, 0] - soh) <= SOH_RADIUS) & (np.abs(ages[indexes] - age) <= AGE_RADIUS)

    def supported(self, state, age, templates):
        soh = state[0]
        if soh < MIN_SOH:
            return np.zeros(len(PRESETS), dtype=bool)
        nearby = self.train[self.nearby(soh, age, self.train)]
        counts = [len(set(self.train_cells[nearby[self.groups[nearby] == action]])) for action in PRESETS]
        return self.nearby(soh, age, templates) & (np.asarray(counts) >= MIN_CELLS)

    def predict(self, state, age, template_ids):
        tiled = np.repeat(np.asarray(state)[None, :], len(template_ids), axis=0)
        return transition(self.model, tiled, np.full(len(template_ids), age), profiles[template_ids])

    def build_case(self, origin_index):
        state, age = initial[origin_index].copy(), int(ages[origin_index])
        if state[0] < MIN_SOH or not np.isfinite(state[1]):
            return None
        nearby = self.train[self.nearby(state[0], age, self.train)]
        templates = []
        for action in PRESETS:
            rows = nearby[self.groups[nearby] == action]
            if len(set(self.train_cells[rows])) < MIN_CELLS:
                return None
            distance = np.abs(initial[rows, 0] - state[0]) / SOH_RADIUS + np.abs(ages[rows] - age) / AGE_RADIUS
            templates.append(rows[distance.argmin()])
        templates = np.asarray(templates)
        middle = []
        forecast_state = state.copy()
        for step in range(HORIZON):
            if not self.supported(forecast_state, age + 10 * step, templates)[1]:
                return None
            change, wh = self.predict(forecast_state, age + 10 * step, templates[1:2])[0]
            middle.append(float(wh))
            forecast_state = np.r_[forecast_state[0] + change, forecast_state[:2]]
        if min(middle) <= 0:
            return None
        return {"index": origin_index, "state": state, "age": age, "templates": templates,
                "reserve": np.asarray(middle), "budget": ENERGY_FRACTION * sum(middle)}

    def select_mask(self, state, age, case, step, credited):
        allowed = self.supported(state, age, case["templates"])
        estimates = self.predict(state, age, case["templates"])
        feasible = allowed & (credited + estimates[:, 1] + case["reserve"][step + 1:].sum() >= case["budget"])
        if allowed.any() and not feasible.any():
            recovery = int(np.argmax(np.where(allowed, estimates[:, 1], -np.inf)))
            feasible[recovery] = True
            return feasible, estimates, True
        return feasible, estimates, False


class DischargeEnv(gym.Env):
    def __init__(self, study, cases):
        super().__init__()
        self.study, self.cases = study, cases
        self.action_space = gym.spaces.Discrete(len(PRESETS))
        self.observation_space = gym.spaces.Box(-np.inf, np.inf, shape=(12,), dtype=np.float32)

    def observation(self):
        soh = self.state[0]
        remaining = self.case["budget"] - self.credited
        reserve = self.case["reserve"][self.turn:].sum()
        drift = soh - self.state[1]
        obs = np.r_[(soh - 90) / 10, drift, (self.age - 80) / 100, self.turn / HORIZON,
                    (remaining - reserve) / 10, self.case["budget"] / 60,
                    self.estimates[:, 0], (self.estimates[:, 1] - 55) / 15]
        return obs.astype(np.float32)

    def update_mask(self):
        self.mask, self.estimates, self.recovery = self.study.select_mask(
            self.state, self.age, self.case, self.turn, self.credited)

    def info(self):
        return dict(action_mask=self.mask.copy(), soh=float(self.state[0]), wh=float(self.credited),
                    budget=self.case["budget"], guard_recoveries=self.guard_recoveries,
                    unsupported_stop=self.unsupported_stop)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = options or {}
        self.case = options.get("case") or self.cases[int(self.np_random.integers(len(self.cases)))]
        if "noise" in options:
            self.noise = options["noise"]
        else:
            self.noise = self.study.noise[int(self.np_random.integers(len(self.study.noise)))]
        self.state, self.age = self.case["state"].copy(), self.case["age"]
        self.turn, self.credited, self.guard_recoveries, self.unsupported_stop = 0, 0.0, 0, False
        self.update_mask()
        assert self.mask.any()
        return self.observation(), self.info()

    def step(self, action):
        if not self.mask[action]:
            raise ValueError("The requested action is masked by the shared controller guard.")
        if self.recovery:
            self.guard_recoveries += 1
        change, wh = self.estimates[action] + self.noise[self.turn]
        delivered = max(0.0, float(wh))
        self.state = np.r_[self.state[0] + change, self.state[:2]]
        self.turn += 1
        self.age += 10
        self.credited += delivered
        finished = self.turn == HORIZON
        if not finished:
            self.update_mask()
            self.unsupported_stop = not self.mask.any()
        else:
            self.mask = np.zeros(len(PRESETS), dtype=bool)
        done = finished or self.unsupported_stop
        reward = float(change) - (3.0 * max(0, self.case["budget"] - self.credited) if done else 0)
        if self.unsupported_stop:
            reward -= 5
        return self.observation(), reward, done, False, self.info()


def all_cases(study):
    starts = chains[HORIZON][:, 0]
    training, testing, rejected = [], [], []
    for index in starts:
        case = study.build_case(index)
        test = cell_ids[index] == study.held_out
        if case is not None:
            (testing if test else training).append(case)
        elif test:
            rejected.append(index)
    if not training:
        raise ValueError(f"No supported training starts for {study.held_out}.")
    return training, testing, rejected


class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(12, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh())
        self.actor, self.critic = nn.Linear(32, 3), nn.Linear(32, 1)

    def forward(self, observations, mask):
        hidden = self.trunk(observations)
        logits = self.actor(hidden).masked_fill(~mask, -1e9)
        return Categorical(logits=logits), self.critic(hidden).squeeze(-1)


def choose_lookahead(env):
    study, case = env.study, env.case

    def utility(state, age, step, credited):
        if step == HORIZON:
            return float(state[0] - 3 * max(0, case["budget"] - credited))
        mask, estimates, _ = study.select_mask(state, age, case, step, credited)
        if not mask.any():
            return float(state[0] - 5 - 3 * max(0, case["budget"] - credited))
        return max(follow(state, age, step, credited, estimates[action]) for action in np.flatnonzero(mask))

    def follow(state, age, step, credited, estimate):
        change, wh = estimate
        next_state = np.r_[state[0] + change, state[:2]]
        return utility(next_state, age + 10, step + 1, credited + max(0, wh))

    return min(np.flatnonzero(env.mask), key=lambda action:
               (-follow(env.state, env.age, env.turn, env.credited, env.estimates[action]), int(action)))


def load_ppo():
    weights = torch.load(io.BytesIO(artifact("ppo_pilot_weights.pt", pilot)), map_location="cpu", weights_only=True)
    result = {}
    for cell in cells:
        actors = {}
        for seed in PPO_SEEDS:
            actor = PolicyNet()
            actor.load_state_dict(weights[f"{cell}/seed-{seed}"])
            actors[seed] = actor.eval()
        result[cell] = actors
    return result


def check_pilot_config():
    config = json.loads(artifact("run_config.json", pilot))
    expected = dict(horizon_tests=HORIZON, seeds=list(PPO_SEEDS), disturbance_paths_per_start=DISTURBANCES,
                    evaluation_seed=SEED, budget_fraction=ENERGY_FRACTION, energy_alpha=ENERGY_ALPHA,
                    min_soh=MIN_SOH, support_soh_radius_pp=SOH_RADIUS, support_age_radius=AGE_RADIUS,
                    support_min_training_cells=MIN_CELLS)
    if any(config.get(key) != value for key, value in expected.items()):
        raise ValueError("Notebook 20 artifacts use a different simulator or evaluation configuration.")


check_pilot_config()
ppo_actors = load_ppo()

class BatchedStudy:
    def __init__(self, study):
        self.study = study
        train = study.train
        self.training_soh = initial[train, 0]
        self.training_age = ages[train]
        cells_count = len(cells)
        indicator = study.groups[train] * cells_count + study.train_cells[train]
        self.membership = np.eye(len(PRESETS) * cells_count)[indicator]

    def mask_and_estimates(self, states, age, templates, reserve, budget, credited, turn):
        soh = states[:, 0]
        supported = []
        for start in range(0, len(soh), 2048):
            stop = start + 2048
            area = ((np.abs(soh[start:stop, None] - self.training_soh) <= SOH_RADIUS)
                    & (np.abs(age[start:stop, None] - self.training_age) <= AGE_RADIUS))
            counts = (area @ self.membership > 0).reshape(-1, len(PRESETS), len(cells)).sum(axis=2)
            supported.append(counts >= MIN_CELLS)
        proximate = ((np.abs(initial[templates, 0] - soh[:, None]) <= SOH_RADIUS)
                     & (np.abs(ages[templates] - age[:, None]) <= AGE_RADIUS))
        allowed = np.concatenate(supported) & proximate & (soh[:, None] >= MIN_SOH)
        estimates = transition(self.study.model, np.repeat(states, len(PRESETS), axis=0),
                               np.repeat(age, len(PRESETS)), profiles[templates.ravel()])
        estimates = estimates.reshape(-1, len(PRESETS), 2)
        possible_energy = credited[:, None] + estimates[:, :, 1] + reserve[:, turn+1:].sum(axis=1)[:, None]
        mask = allowed & (possible_energy >= budget[:, None])
        recover = allowed.any(axis=1) & ~mask.any(axis=1)
        if recover.any():
            chosen = np.argmax(np.where(allowed[recover], estimates[recover, :, 1], -np.inf), axis=1)
            mask[np.flatnonzero(recover), chosen] = True
        return mask, estimates, recover


def many_observations(states, age, credited, budget, reserve, turn, estimates):
    soh = states[:, 0]
    drift = soh - states[:, 1]
    remaining = (budget - credited - reserve[:, turn:].sum(axis=1)) / 10
    return np.column_stack(((soh - 90) / 10, drift, (age - 80) / 100,
                            np.full(len(age), turn / HORIZON), remaining, budget / 60,
                            estimates[:, :, 0], (estimates[:, :, 1] - 55) / 15)).astype(np.float32)


def check_batched_model(env, batched, case, noise):
    observation, _ = env.reset(options={"case": case, "noise": noise})
    for step in range(HORIZON):
        mask, estimates, recovered = batched.mask_and_estimates(
            env.state[None], np.array([env.age]), case["templates"][None],
            case["reserve"][None], np.array([case["budget"]]), np.array([env.credited]), step)
        np.testing.assert_array_equal(mask[0], env.mask)
        np.testing.assert_allclose(estimates[0], env.estimates, atol=1e-9, rtol=0)
        assert recovered[0] == env.recovery
        stacked = many_observations(env.state[None], np.array([env.age]), np.array([env.credited]),
                                   np.array([case["budget"]]), case["reserve"][None], step, estimates)
        np.testing.assert_allclose(stacked[0], observation, atol=1e-7, rtol=0)
        observation, _, done, _, _ = env.step(int(np.flatnonzero(env.mask)[0]))
        if done:
            break


def collect_batch(training, study, batched, rng, mask_log=None):
    tags = np.repeat(np.arange(len(training)), TRAIN_BRANCHES)
    noise = study.noise[rng.integers(len(study.noise), size=len(tags))]
    draws = np.arange(len(tags))
    cases_templates = np.stack([case["templates"] for case in training])
    cases_reserve = np.stack([case["reserve"] for case in training])
    cases_budget = np.asarray([case["budget"] for case in training])
    states = np.stack([case["state"] for case in training])[tags]
    age = np.asarray([case["age"] for case in training])[tags]
    credit = np.zeros(len(tags))
    records = []
    for turn in range(HORIZON):
        templates, reserve, budget = cases_templates[tags], cases_reserve[tags], cases_budget[tags]
        mask, estimates, _ = batched.mask_and_estimates(states, age, templates, reserve, budget, credit, turn)
        observation = many_observations(states, age, credit, budget, reserve, turn, estimates)
        root, action = np.nonzero(mask)
        if not len(root):
            raise ValueError(f"No supported FQI training actions in stage {turn + 1}.")
        next_tags, next_draws, next_age = tags[root], draws[root], age[root] + 10
        delta = estimates[root, action] + noise[next_draws, turn]
        new_state = np.column_stack((states[root, 0] + delta[:, 0], states[root, :2]))
        new_credit = credit[root] + np.maximum(0, delta[:, 1])
        final = turn == HORIZON - 1
        if final:
            next_mask = np.zeros((len(root), len(PRESETS)), dtype=bool)
            next_observation = np.zeros_like(observation[root])
        else:
            next_mask, next_estimates, _ = batched.mask_and_estimates(
                new_state, next_age, templates[root], reserve[root], budget[root], new_credit, turn + 1)
            next_observation = many_observations(new_state, next_age, new_credit,
                                                  budget[root], reserve[root], turn + 1, next_estimates)
        done = ~next_mask.any(axis=1)
        penalty = np.where(done, 3 * np.maximum(0, budget[root] - new_credit), 0)
        reward = delta[:, 0] - penalty - (0 if final else 5 * done)
        records.append((observation[root], action, reward, next_observation, next_mask))
        if mask_log is not None:
            mask_log.append(mask[root])
        viable = ~done
        tags, draws, states, age, credit = (next_tags[viable], next_draws[viable], new_state[viable],
                                            next_age[viable], new_credit[viable])
    return records


def action_columns(observation, action):
    return np.column_stack((observation, np.eye(len(PRESETS))[action]))


def fit_fqi(records):
    values, notes = {}, []
    for step in reversed(range(HORIZON)):
        obs, actions, reward, following, legal = records[step]
        value = reward.copy()
        j, option = np.nonzero(legal)
        if len(j):
            estimate = values[step + 1].predict(action_columns(following[j], option))
            next_value = np.full(len(reward), -np.inf)
            np.maximum.at(next_value, j, estimate)
            alive = legal.any(axis=1)
            value[alive] += next_value[alive]
        assert np.isfinite(value).all()
        tree = ExtraTreesRegressor(n_estimators=64, max_depth=6, min_samples_leaf=5,
                                   n_jobs=1, random_state=SEED + step)
        tree.fit(action_columns(obs, actions), value)
        values[step] = tree
        notes.append((step + 1, len(reward), int((~legal.any(axis=1)).sum()), *np.bincount(actions, minlength=3)))
    return values, notes


def fqi_action(fitted, env, observation):
    mask = env.mask
    actions = np.flatnonzero(mask)
    inputs = action_columns(np.repeat(observation[None], len(actions), axis=0), actions)
    return int(actions[fitted[env.turn].predict(inputs).argmax()])

def baseline_action(env, policy):
    if policy == "lookahead":
        return choose_lookahead(env)
    preferred = 0 if policy == "rule" and env.state[0] < 90 else 1
    if env.mask[preferred]:
        return preferred
    return int(np.argmax(np.where(env.mask, env.estimates[:, 1], -np.inf)))

def verify_baselines(episodes, cases):
    keys = ["cell_id", "start_cycle", "draw", "policy", "seed"]
    reference = read_table("test_episodes", pilot, dtype={"actions": "string"})
    reference = reference.loc[reference.policy.isin(["lookahead", "ppo"])].set_index(keys, verify_integrity=True).sort_index()
    restored = episodes.loc[episodes.policy.isin(["lookahead", "ppo"])].set_index(keys, verify_integrity=True).sort_index()
    pd.testing.assert_index_equal(reference.index, restored.index)
    for name in ("soh_loss_pp", "budget_wh", "delivered_wh"):
        np.testing.assert_allclose(reference[name], restored[name], rtol=0, atol=TOLERANCE)
    for name in ("budget_hit", "unsupported_stop", "guard_recoveries", "intervals_completed", "actions"):
        np.testing.assert_array_equal(reference[name], restored[name])
    assert cases.test_starts.sum() == 133
    assert cases.test_starts.sum() + cases.excluded_starts.sum() == len(windows[3].cells)

In [9]:
def actor_or_critic(width):
    return nn.Sequential(nn.Linear(width, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))


def legal_distribution(logits, mask):
    assert mask.any(dim=-1).all()
    log = torch.log_softmax(logits.masked_fill(~mask, -1e9), dim=-1)
    probs = torch.where(mask, log.exp(), 0)
    return probs, torch.where(mask, log, 0)


class DiscreteSAC:
    def __init__(self, width):
        self.actor = actor_or_critic(width)
        self.q1, self.q2 = actor_or_critic(width), actor_or_critic(width)
        self.target1, self.target2 = actor_or_critic(width), actor_or_critic(width)
        self.target1.load_state_dict(self.q1.state_dict())
        self.target2.load_state_dict(self.q2.state_dict())
        self.log_alpha = nn.Parameter(torch.tensor(-2.3025851))

    def choose(self, observation, mask):
        return int(self.choose_batch(np.atleast_2d(observation), np.atleast_2d(mask))[0])

    def choose_batch(self, obs, mask):
        with torch.no_grad():
            logits = self.actor(torch.as_tensor(obs, dtype=torch.float32))
            return logits.masked_fill(~torch.as_tensor(mask), -1e9).argmax(dim=1).numpy()

    def fit(self, raw, seed):
        optimizer = torch.optim.Adam(self.actor.parameters(), lr=.0003)
        critic = torch.optim.Adam(list(self.q1.parameters())+list(self.q2.parameters()), lr=.0003)
        entropy_optimizer = torch.optim.Adam([self.log_alpha], lr=.0003)
        observed, masks, action, rewards, next_obs, next_masks, terminal = [
            torch.as_tensor(value, dtype=dtype) for value, dtype in
            zip(raw, (torch.float32, torch.bool, torch.long, torch.float32,
                      torch.float32, torch.bool, torch.float32))
        ]
        rng = np.random.default_rng(seed)
        records = []
        for step in range(1, SAC_UPDATES + 1):
            ix = rng.integers(len(observed), size=SAC_BATCH)
            x, valid, a, r, y, future, done = (
                observed[ix], masks[ix], action[ix], rewards[ix], next_obs[ix], next_masks[ix], terminal[ix])
            with torch.no_grad():
                p, logp = legal_distribution(self.actor(y), future)
                future_min = torch.minimum(self.target1(y), self.target2(y))
                estimated = r + (1-done) * (p * (future_min-self.log_alpha.exp()*logp)).sum(dim=1)
            q1 = self.q1(x).gather(1, a[:, None]).squeeze(1)
            q2 = self.q2(x).gather(1, a[:, None]).squeeze(1)
            qloss = nn.functional.smooth_l1_loss(q1, estimated) + nn.functional.smooth_l1_loss(q2, estimated)
            critic.zero_grad()
            qloss.backward()
            critic.step()

            p, logp = legal_distribution(self.actor(x), valid)
            with torch.no_grad():
                minimum = torch.minimum(self.q1(x), self.q2(x))
            aloss = (p * (self.log_alpha.detach().exp()*logp-minimum)).sum(dim=1).mean()
            optimizer.zero_grad()
            aloss.backward()
            optimizer.step()

            entropy = -(p * logp).sum(dim=1).detach()
            alpha_loss = (self.log_alpha * (entropy - .7*valid.sum(dim=1).float().log())).mean()
            entropy_optimizer.zero_grad()
            alpha_loss.backward()
            entropy_optimizer.step()
            with torch.no_grad():
                self.log_alpha.clamp_(-7, 0)
                for source, target in ((self.q1, self.target1), (self.q2, self.target2)):
                    for a, b in zip(source.parameters(), target.parameters()):
                        b.lerp_(a, TAU)
            if step % 250 == 0:
                if not torch.isfinite(qloss + aloss):
                    raise ValueError("SAC training produced a nonfinite loss.")
                records.append((step, float(qloss.detach()), float(aloss.detach()), float(self.log_alpha.exp().detach())))
        self.actor.eval()
        return records



def pack_sac(records, masks):
    x, action, reward, following, next_legal = [np.concatenate(column) for column in zip(*records)]
    mask = np.concatenate(masks)
    assert mask[np.arange(len(x)), action].all()
    terminal = ~next_legal.any(axis=1)
    next_legal[terminal, 0] = True  # Discarded by the terminal Bellman mask.
    assert all(np.isfinite(values).all() for values in (x, reward, following))
    return x, mask, action, reward, following, next_legal, terminal


def fit_sac_agents(records, masks):
    data = pack_sac(records, masks)
    actors, history = {}, []
    for seed in SEEDS:
        torch.manual_seed(seed)
        agent = DiscreteSAC(data[0].shape[1])
        history.extend((seed, *row) for row in agent.fit(data, seed))
        actors[seed] = agent
    return actors, history


probabilities, _ = legal_distribution(torch.ones((2, 3)), torch.tensor([[1, 0, 1], [0, 1, 0]], dtype=torch.bool))
torch.testing.assert_close(probabilities, torch.tensor([[.5, 0, .5], [0, 1, 0]]))

In [10]:
class ForecastPlanner:
    def __init__(self, study, batched, planning_paths):
        self.study = study
        self.batched = batched
        self.paths = planning_paths
        self.models = forecast_models[study.held_out]
        self.cache = {}

    def simulation(self, env, sequences, noise):
        """Apply the frozen guard along each proposed sequence on each planning path."""
        turns = env.turn
        n_plans, remaining = sequences.shape
        n_paths = len(noise)
        count = n_plans*n_paths
        candidates = np.repeat(sequences, n_paths, axis=0)
        disturbances = np.tile(noise, (n_plans, 1, 1))
        state = np.repeat(env.state[None], count, axis=0)
        age = np.full(count, env.age)
        credit = np.full(count, env.credited)
        budget = np.full(count, env.case["budget"])
        templates = np.repeat(env.case["templates"][None], count, axis=0)
        reserve = np.repeat(env.case["reserve"][None], count, axis=0)
        valid = np.ones(count, dtype=bool)
        for step in range(remaining):
            copies = np.flatnonzero(valid)
            if not len(copies):
                break
            mask, estimates, _ = self.batched.mask_and_estimates(
                state[copies], age[copies], templates[copies], reserve[copies], budget[copies], credit[copies], turns+step)
            action = candidates[copies, step]
            legal = mask[np.arange(len(copies)), action]
            valid[copies[~legal]] = False
            advance = copies[legal]
            change = estimates[np.flatnonzero(legal), action[legal]] + disturbances[advance, step]
            prior = state[advance]
            state[advance] = np.column_stack((prior[:, 0]+change[:, 0], prior[:, :2]))
            credit[advance] += np.maximum(0., change[:, 1])
            age[advance] += BLOCK
        shape = (n_plans, n_paths)
        return state[:, 0].reshape(shape), credit.reshape(shape), valid.reshape(shape)

    def proposed_forecasts(self, env, sequences, history_soh, history_wh, nominal_soh, nominal_wh):
        count, horizon = sequences.shape
        origin = env.case["index"]
        inputs = ForecastInputs(
            np.repeat(history_soh[None], count, axis=0),
            np.repeat(history_wh[None], count, axis=0),
            np.repeat(curve_snapshot[origin:origin+1], count, axis=0),
            np.full(count, env.age),
            commands[env.case["templates"][sequences]],
        )
        saved = self.models[horizon]
        choices = saved["choices"]
        soh_names = tuple(name for name in required_names(choices["soh"]) if name != "baseline")
        wh_names = tuple(name for name in required_names(choices["wh"]) if name != "baseline")
        soh, wh = candidate_predictions(saved["fitted"], inputs, horizon, soh_names, wh_names)
        # Legacy forecast-only Ridge inputs include recorded block context. Within simulation,
        # a baseline head uses the original simulator projection and observed simulated state.
        soh["baseline"] = nominal_soh
        wh["baseline"] = nominal_wh-env.credited
        return use_choice(choices["soh"], soh), env.credited+use_choice(choices["wh"], wh)

    def choose(self, env, history_soh, history_wh, risk):
        recent = np.r_[np.nan_to_num(history_soh, nan=-1.), np.nan_to_num(history_wh, nan=-1.), env.credited]
        key = env.case["index"], env.turn, risk, recent.tobytes()
        cached = self.cache.get(key)
        if cached is not None:
            return cached

        remaining = HORIZON-env.turn
        sequences = np.asarray([plan for plan in product(PRESETS, repeat=remaining) if env.mask[plan[0]]], dtype=int)
        base_soh, base_wh, base_valid = self.simulation(env, sequences, np.zeros((1, remaining, 2)))
        soh, wh = self.proposed_forecasts(env, sequences, history_soh, history_wh, base_soh[:, 0], base_wh[:, 0])
        if risk:
            draws_soh, draws_wh, valid = self.simulation(env, sequences, self.paths[:, env.turn:])
            outcomes = draws_soh+(soh-base_soh[:, 0])[:, None]
            energy = draws_wh+(wh-base_wh[:, 0])[:, None]
            hits = valid & base_valid & (energy >= env.case["budget"])
            feasible = np.flatnonzero(hits.mean(axis=1) >= PLANNING_RISK)
            losses = env.state[0]-outcomes+5.*~valid
            tail = np.sort(losses, axis=1)[:, -int(np.ceil(losses.shape[1]*TAIL_FRACTION)):].mean(axis=1)
            rank = np.lexsort((feasible, losses[feasible].mean(axis=1), tail[feasible]))
        else:
            feasible = np.flatnonzero(base_valid[:, 0] & (wh >= env.case["budget"]))
            rank = np.lexsort((feasible, -wh[feasible], -soh[feasible]))
        decision = ((int(sequences[feasible[rank[0]], 0]), False) if len(feasible)
                    else (int(choose_lookahead(env)), True))
        # Risk and nominal planning must remain separate from the eventual scoring draw.
        self.cache[key] = decision
        return decision

    def attainable(self, env, hindsight):
        # Per-episode lower bound among complete paths that hit the shared energy target.
        sequences = np.asarray([plan for plan in product(PRESETS, repeat=HORIZON) if env.mask[plan[0]]], dtype=int)
        terminal, energy, valid = self.simulation(env, sequences, hindsight[None])
        possible = np.flatnonzero(valid[:, 0] & (energy[:, 0] >= env.case["budget"]))
        if not len(possible):
            return np.nan, ""
        choice = int(possible[terminal[possible, 0].argmax()])
        return float(env.state[0]-terminal[choice, 0]), "".join(map(str, sequences[choice]))

In [11]:
def controller_episode(env, case, noise, policy, agent, fitted, planner):
    obs, info = env.reset(options={"case": case, "noise": noise})
    history_soh = health_history[case["index"]].copy()
    history_wh = wh_history[case["index"]].copy()
    actions, fallback_count = [], 0
    while True:
        if policy == "ppo":
            with torch.inference_mode():
                distribution, _ = agent(torch.as_tensor(obs), torch.as_tensor(info["action_mask"]))
                action = int(distribution.logits.argmax())
        elif policy == "sac":
            action = agent.choose(obs, env.mask)
        elif policy == "fqi":
            action = fqi_action(fitted, env, obs)
        elif policy in {"mpc", "risk"}:
            action, fell_back = planner.choose(env, history_soh, history_wh, policy == "risk")
            fallback_count += fell_back
        else:
            action = int(baseline_action(env, policy))
        assert env.mask[action]
        actions.append(str(action))
        prior_credit = info["wh"]
        obs, _, done, _, info = env.step(action)
        history_soh = np.r_[info["soh"], history_soh[:-1]]
        history_wh = np.r_[info["wh"]-prior_credit, history_wh[:-1]]
        if done:
            break
    return dict(soh_loss_pp=case["state"][0]-info["soh"], delivered_wh=info["wh"], budget_wh=info["budget"],
                budget_hit=info["wh"] >= info["budget"] and not info["unsupported_stop"],
                guard_recoveries=info["guard_recoveries"], unsupported_stop=info["unsupported_stop"],
                intervals_completed=len(actions), actions="".join(actions), planning_fallbacks=fallback_count)


def controller_trials():
    outcomes, sizes, sac_log, oracle, agents = [], [], [], [], {}
    rng = np.random.default_rng(SEED)
    for fold, cell in enumerate(cells):
        study = ScheduleStudy(cell, models[cell], disturbances[cell])
        train, test, rejected = all_cases(study)
        assert all(cell_ids[item["index"]] != cell for item in train)
        assert all(cell_ids[item["index"]] == cell for item in test)
        assert all(cell_ids[index] != cell for item in train+test for index in item["templates"])
        env = DischargeEnv(study, train)
        batch = BatchedStudy(study)
        check_batched_model(env, batch, train[0], study.noise[0])
        mask_log = []
        generator = np.random.default_rng(np.random.SeedSequence([SEED, fold, 1]))
        paths = collect_batch(train, study, batch, generator, mask_log)
        fitted, _ = fit_fqi(paths)
        sac, log = fit_sac_agents(paths, mask_log)
        agents.update({f"{cell}/seed-{seed}": agent.actor.state_dict() for seed, agent in sac.items()})
        sac_log.extend((cell, *row) for row in log)
        planning = np.random.default_rng(np.random.SeedSequence([SEED, fold, 2]))
        samples = study.noise[planning.integers(len(study.noise), size=PLANNING_PATHS)]
        planner = ForecastPlanner(study, batch, samples)
        sizes.append((cell, len(train), len(test), len(rejected), sum(len(array) for array in mask_log)))
        choices = {name: [(0, None)] for name in POLICIES}
        choices.update(ppo=list(ppo_actors[cell].items()), sac=list(sac.items()))
        first_case = test[0]
        env.reset(options={"case": first_case, "noise": study.noise[0]})
        origin = first_case["index"]
        for risk in (False, True):
            guard = planner.choose(env, health_history[origin], wh_history[origin], risk)
            env.noise = np.ones_like(study.noise[0])*900.
            planner.cache.clear()
            assert guard == planner.choose(env, health_history[origin], wh_history[origin], risk)
        for case in test:
            start = int(data.iloc[case["index"]].start_cycle)
            noises = study.noise[rng.integers(len(study.noise), size=DISTURBANCES)]
            for draw, noise in enumerate(noises):
                for evaluation, scale in (("original", 1.), ("stress", STRESS_SCALE)):
                    conditioned = noise*scale
                    env.reset(options={"case": case, "noise": conditioned})
                    loss, sequence = planner.attainable(env, conditioned)
                    oracle.append((evaluation, cell, start, draw, loss, sequence))
                    for policy, runs in choices.items():
                        for seed, agent in runs:
                            results = controller_episode(env, case, conditioned, policy, agent, fitted, planner)
                            outcomes.append(dict(cell_id=cell, start_cycle=start, evaluation=evaluation,
                                                 draw=draw, policy=policy, seed=seed, **results))
        print(f"Controller fold {cell}: {len(test)} held-out starts and {sizes[-1][-1]:,} legal training transitions.", flush=True)
    return (
        pd.DataFrame(outcomes),
        pd.DataFrame(sizes, columns=["cell_id", "train_starts", "test_starts", "excluded_starts", "sac_transitions"]),
        pd.DataFrame(sac_log, columns=["cell_id", "seed", "updates", "critic_loss", "actor_loss", "temperature"]),
        pd.DataFrame(oracle, columns=["evaluation", "cell_id", "start_cycle", "draw", "attainable_loss_pp", "sequence"]),
        agents,
    )


def verify_controller_trials(episodes, cases):
    original = episodes.loc[episodes.evaluation.eq("original")]
    verify_baselines(original, cases)
    old = read_table("controller_episodes", benchmark, dtype={"actions": "string"})
    keys = ["cell_id", "start_cycle", "draw", "seed", "policy"]
    previous = old.loc[old.policy.eq("fqi")].set_index(keys, verify_integrity=True).sort_index()
    replayed = original.loc[original.policy.eq("fqi")].set_index(keys, verify_integrity=True).sort_index()
    pd.testing.assert_index_equal(previous.index, replayed.index)
    np.testing.assert_array_equal(previous.actions, replayed.actions)
    for name in ("soh_loss_pp", "delivered_wh", "budget_wh"):
        np.testing.assert_allclose(previous[name], replayed[name], rtol=0, atol=1e-9)
    scenario_keys = ["evaluation", "cell_id", "start_cycle", "draw"]
    grouped = episodes.groupby(scenario_keys)
    assert len(grouped) == 2*133*DISTURBANCES
    assert grouped.size().eq(12).all()
    assert grouped.budget_wh.nunique().eq(1).all()
    assert episodes.groupby(["cell_id", "start_cycle"]).budget_wh.nunique().eq(1).all()
    print("Reproduced the original Lookahead, PPO, and FQI trajectories. Verified paired budgets and starts.")



CONTROLLER_CACHED_TABLES = ("controller_episodes", "controller_cases", "controller_metrics",
                            "controller_seed_metrics", "controller_comparisons", "sac_training", "oracle_headroom")


def controller_cache():
    required = ("run_config.json", "sac_actor_weights.pt", *(name+".csv" for name in CONTROLLER_CACHED_TABLES))
    if RERUN_CONTROLLER_BENCHMARK:
        return None
    for candidate in (output, output.with_suffix(".zip")):
        if candidate.is_dir() and not all((candidate / name).is_file() for name in required):
            continue
        if candidate.is_file():
            with ZipFile(candidate) as archive:
                members = [PurePosixPath(member).name for member in archive.namelist()]
            if any(members.count(name) != 1 for name in required):
                continue
        elif not candidate.is_dir():
            continue
        saved = json.loads(artifact("run_config.json", candidate))
        expected = dict(starts=133, disturbance_seed=SEED, disturbance_paths=DISTURBANCES,
                        stress_scale=STRESS_SCALE, energy_budget_fraction=ENERGY_FRACTION,
                        seeds=list(SEEDS), sac_updates=SAC_UPDATES,
                        conservative_planning_paths=PLANNING_PATHS, planning_energy_probability=PLANNING_RISK)
        if all(saved.get(key) == value for key, value in expected.items()):
            return candidate
    return None


controller_archive = controller_cache()
if controller_archive is None:
    print("Running the original controller benchmark.", flush=True)
    with threadpool_limits(limits=1):
        controller_episodes, controller_cases, sac_training, oracle_cases, actor_weights = controller_trials()
else:
    controller_episodes, controller_cases, sac_training, oracle_cases = (
        read_table(name, controller_archive, float_precision="round_trip",
                   dtype={"actions": "string", "sequence": "string"})
        for name in ("controller_episodes", "controller_cases", "sac_training", "oracle_headroom")
    )
    actor_weights = torch.load(io.BytesIO(artifact("sac_actor_weights.pt", controller_archive)),
                               map_location="cpu", weights_only=True)
    print("Reusing the saved original controller benchmark.", flush=True)
assert set(actor_weights) == {f"{cell}/seed-{seed}" for cell in cells for seed in SEEDS}
verify_controller_trials(controller_episodes, controller_cases)


Reusing the saved original controller benchmark.
Reproduced the original Lookahead, PPO, and FQI trajectories. Verified paired budgets and starts.


In [12]:
def refinement_report():
    text = [
        "## Revised forecasts on the same held-out windows", "",
        "Each error cell shows original Ridge or Direct Ridge, the previous Notebook 23 recommendation, and the revised "
        "recommendation. The proposed correction is promoted when mean, worst-decile and maximum error all match or "
        "improve on the previous recommendation and at least one decreases.", "",
        "| Setting | Target | Windows | Mean absolute error | Worst 10% mean | Maximum error | Recommendation |",
        "| --- | --- | ---: | ---: | ---: | ---: | --- |",
    ]
    for (mode, cycles, target), frame in refinement_metrics.groupby(["mode", "cycles", "target"]):
        scores = frame.set_index("role")
        setting = f"{cycles} ahead at the start" if mode == "fixed" else f"Refresh after {cycles} of 30"
        unit = "SOH (pp)" if target == "soh" else "Delivered energy (Wh)"
        change = []
        for field in ("mae", "worst_10", "max_error"):
            change.append(" → ".join(f"{scores.at[role, field]:.3f}" for role in ("original", "incumbent", "recommended")))
        verdict = "Revised" if scores.at["recommended", "recommendation"] == "revision" else "Previous"
        text.append(f"| {setting} | {unit} | {scores.at['recommended', 'windows']} | {' | '.join(change)} | {verdict} |")
    text.extend(["", "### Proposed correction compared with the previous recommendation", "",
        "| Setting | Target | Mean error reduction | Worst 10% reduction | Maximum reduction | Paired MAE change 95% interval |",
        "| --- | --- | ---: | ---: | ---: | ---: |"])
    for row in refinement_metrics.loc[refinement_metrics.role.eq("proposed")].itertuples(index=False):
        setting = f"{row.cycles} ahead at the start" if row.mode == "fixed" else f"Refresh after {row.cycles} of 30"
        unit = "SOH (pp)" if row.target == "soh" else "Delivered energy (Wh)"
        text.append(f"| {setting} | {unit} | {row.mae_reduction_pct:+.1f}% | {row.tail_reduction_pct:+.1f}% | "
                    f"{row.max_reduction_pct:+.1f}% | {row.mae_ci95_low:+.3f} to {row.mae_ci95_high:+.3f} |")
    text.extend(["", "Weights and correction limits were selected within the training batteries. Inner candidate "
                 "predictions and Wh bias-training residuals were cross-fitted by battery. Rolling energy predictions "
                 "add the energy measured before the update to the remaining energy forecast. These eight batteries "
                 "also informed earlier screening.", "",
                 "The 50/50 blends appear in the CSV as exploratory comparisons. The CSVs also include fold settings, "
                 "worst cases, and the subset whose SOH was below 90% when the forecast was made.", "",
                 "The controller section reports the original Notebook 23 benchmark with the earlier planner forecasts.", ""])
    return text


def controller_scores():
    labels = ["soh_loss_pp", "delivered_wh", "budget_hit", "unsupported_stop"]
    battery = ["evaluation", "policy", "cell_id"]
    groups = controller_episodes.groupby(battery)
    by_battery = groups[labels].mean()
    by_battery["tail_loss_pp"] = groups.soh_loss_pp.agg(upper_tail)
    aggregate = by_battery.groupby(battery[:2]).mean()
    details = controller_episodes.assign(shortfall=np.maximum(0., controller_episodes.budget_wh-controller_episodes.delivered_wh))
    all_groups = details.groupby(battery[:2])
    aggregate["max_loss_pp"] = all_groups.soh_loss_pp.max()
    aggregate["max_shortfall_wh"] = all_groups.shortfall.max()
    aggregate["episodes"] = all_groups.size()
    rows = []
    for test, group in aggregate.groupby(level="evaluation"):
        original = group.loc[(test, "lookahead")]
        for policy in POLICIES:
            if policy == "lookahead":
                continue
            outcome = group.loc[(test, policy)]
            differences = by_battery.loc[(test, policy)]-by_battery.loc[(test, "lookahead")]
            mean_ci = paired_interval(differences.soh_loss_pp.to_numpy())
            tail_ci = paired_interval(differences.tail_loss_pp.to_numpy())
            mean_reduction = 1-outcome.soh_loss_pp/original.soh_loss_pp
            tail_reduction = 1-outcome.tail_loss_pp/original.tail_loss_pp
            mean_pass = mean_reduction >= ADVANCEMENT and mean_ci[1] < 0 and tail_reduction >= -TOLERANCE
            tail_pass = tail_reduction >= ADVANCEMENT and tail_ci[1] < 0 and mean_reduction >= -TOLERANCE
            loss_pass = (mean_pass or tail_pass) and outcome.max_loss_pp <= original.max_loss_pp+TOLERANCE
            hit_change = outcome.budget_hit-original.budget_hit
            stop_change = outcome.unsupported_stop-original.unsupported_stop
            eligible = loss_pass and hit_change >= -ENERGY_RATE_TOLERANCE-TOLERANCE and stop_change <= TOLERANCE
            rows.append((test, policy, 100*mean_reduction, *mean_ci, 100*tail_reduction, *tail_ci,
                         outcome.max_loss_pp-original.max_loss_pp, 100*hit_change, 100*stop_change, eligible))
    comparison = pd.DataFrame(rows, columns=["evaluation", "policy", "mean_loss_reduction_pct", "mean_ci95_low_pp",
        "mean_ci95_high_pp", "tail_loss_reduction_pct", "tail_ci95_low_pp", "tail_ci95_high_pp",
        "max_loss_change_pp", "budget_hit_change_pp", "stop_change_pp", "qualifies"])
    candidates = comparison.loc[comparison.evaluation.eq("original") & comparison.qualifies, "policy"].tolist()
    primary = aggregate.loc["original"]
    policy = (primary.loc[candidates].sort_values(["tail_loss_pp", "soh_loss_pp"], kind="stable").index[0]
              if candidates else "lookahead")
    aggregate["selected"] = aggregate.index.get_level_values("policy") == policy
    return aggregate.reset_index(), comparison, policy


def per_seed_scores():
    frame = controller_episodes.loc[controller_episodes.policy.isin(["ppo", "sac"])]
    grouping = ["evaluation", "policy", "seed"]
    cellscores = frame.groupby([*grouping, "cell_id"])
    metrics = cellscores[["soh_loss_pp", "budget_hit", "unsupported_stop"]].mean()
    metrics["tail_loss_pp"] = cellscores.soh_loss_pp.agg(upper_tail)
    summary = metrics.groupby(grouping).mean()
    summary["max_loss_pp"] = frame.groupby(grouping).soh_loss_pp.max()
    return summary.reset_index()


def forecast_decisions(metrics, index, preferred):
    rows = []
    for keys, group in metrics.groupby([index, "target"]):
        before = group.loc[group.model.eq("baseline")].iloc[0]
        after = group.loc[group.model.eq(preferred)].iloc[0]
        reductions = (after.mae_improvement_pct, after.tail_improvement_pct, after.max_improvement_pct)
        improve = min(reductions) >= -1e-8 and max(reductions) > 1e-8
        rows.append((*keys, "new" if improve else "baseline", min(reductions) >= 50, *reductions,
                     float(before.mae), float(after.mae)))
    return pd.DataFrame(rows, columns=[index, "target", "recommendation", "all_errors_halved",
                     "mae_reduction_pct", "tail_reduction_pct", "max_reduction_pct", "old_mae", "new_mae"])


def forecast_cases():
    pieces = []
    for name, frame, key in (("fixed", forecast_predictions, "horizon_cycles"),
                              ("rolling", rolling_predictions, "elapsed_cycles")):
        method = "selected" if name == "fixed" else "bias_corrected"
        sub = frame.loc[frame.model.isin(["baseline", method])].copy()
        sub["condition"] = np.where(sub.start_soh < 90, "SOH below 90%", "SOH at least 90%")
        error = np.abs(sub.measured-sub.prediction)
        sub["error"] = error
        table = sub.groupby([key, "target", "model", "condition"]).error.agg(windows="size", mae="mean", max_error="max")
        extra = sub.groupby([key, "target", "model", "condition"]).error.agg(worst_10=upper_tail)
        table["worst_10"] = extra
        locked = table.reset_index().rename(columns={key: "cycles"})
        locked["study"] = name
        pieces.append(locked)
    return pd.concat(pieces, ignore_index=True)


def write_forecast_rows(score, time, mode, decisions, select):
    for value, chunk in score.loc[score.model.isin(["baseline", select])].groupby(time):
        for target, group in chunk.groupby("target"):
            before, after = (group.loc[group.model.eq(name)].iloc[0] for name in ("baseline", select))
            status = decisions.loc[decisions[time].eq(value) & decisions.target.eq(target), "recommendation"].iloc[0]
            units = "SOH (pp)" if target == "soh" else "Delivered energy (Wh)"
            fields = [(before.mae, after.mae, after.mae_improvement_pct),
                      (before.worst_10, after.worst_10, after.tail_improvement_pct),
                      (before.max_error, after.max_error, after.max_improvement_pct)]
            changes = " | ".join(f"{first:.3f} → {last:.3f} ({pct:+.1f}% reduction)" for first, last, pct in fields)
            yield f"| {mode.format(cycles=value)} | {units} | {before.windows} | {changes} | {status.title()} |"


def oracle_summary():
    keys = ["evaluation", "cell_id", "start_cycle", "draw"]
    reference = controller_episodes.loc[controller_episodes.policy.eq("lookahead"), keys+["soh_loss_pp"]]
    compared = oracle_cases.merge(reference, on=keys, validate="one_to_one")
    matched = compared.attainable_loss_pp.notna()
    equal = (compared.soh_loss_pp-compared.attainable_loss_pp).abs() <= TOLERANCE
    return compared.assign(attainable=matched, equal=equal).groupby("evaluation")[["attainable", "equal"]].sum()


def describe():
    lines = [
        "# XJTU discharge forecasts and controller benchmark", "",
        f"**Selected controller under the original simulation is {POLICIES[selected_policy]}.**",
        "Forecasts were scored against recorded reference SOH and integrated Wh. Controller outcomes came from the frozen "
        "Notebook 20 simulator. Results represent modeled outcomes over 30 operating cycles.", "",
        *refinement_report(),
        "## Earlier Notebook 23 forecast benchmark", "",
        "The tables report the earlier baseline followed by the fold-selected forecast. Mean absolute errors give each battery "
        "equal weight. The upper decile pools matched windows. The tables use an identical forecast origin for each within-row "
        "comparison. A positive reduction means lower error.", "",
        "| Setting | Target | Windows | Mean absolute error | Worst 10% mean | Maximum error | Recommendation |",
        "| --- | --- | ---: | ---: | ---: | ---: | --- |",
        *write_forecast_rows(forecast_metrics, "horizon_cycles", "At the start, {cycles} ahead", fixed_decisions, "selected"),
        *write_forecast_rows(rolling_metrics, "elapsed_cycles", "{cycles} elapsed of 30", rolling_decisions, "bias_corrected"),
        "",
        "Fixed forecasts used up to five past reference tests, completed operating energy and a measured reference "
        "discharge curve. Forecasts also used the first prescribed current and duration of each planned discharge cycle. "
        "Energy candidates included cycle-level Ridge, an approximate Kernel Ridge, shallow boosting, a compact GRU and "
        "a variant with a cross-fitted SOH forecast. The SOH catalog included local trend, training-battery matches, "
        "curve-based regression and an R3-transfer model adapted on the Random Walk training batteries.",
        "",
        "The model catalog and blends were selected by nested leave-one-battery-out folds. Fixed-origin recommendations "
        "require all three error metrics to match or improve on the incumbent. The experiment targeted a 50% reduction "
        "in all three fixed-origin error metrics at 30 cycles. The same cohort informed earlier model screening. "
        "The rolling results use actual measurements made before each refreshed prediction. The CSVs include battery-level "
        "predictions, bias correction settings and overlap sensitivity for the age and nearest-battery methods.",
        "", "### Prespecified difficult cases", "",
        "| Target | Battery | Start cycle | Prior absolute error | New absolute error |",
        "| --- | --- | ---: | ---: | ---: |",
    ]
    for target, battery, start in (("soh", "XJTU_RW_battery-3", 307), ("wh", "XJTU_RW_battery-1", 32)):
        sub = forecast_predictions.loc[(forecast_predictions.target == target) &
            (forecast_predictions.cell_id == battery) & (forecast_predictions.start_cycle == start) &
            (forecast_predictions.horizon_cycles == 30)].set_index("model")
        first, second = (abs(sub.loc[key, "measured"]-sub.loc[key, "prediction"]) for key in ("baseline", "selected"))
        lines.append(f"| {target.upper()} | {battery} | {start} | {first:.3f} | {second:.3f} |")
    lines.extend(["", "## Controller outcomes", "",
        "All policies shared 133 starts, eight disturbances per start, the 90% energy budget and the original action guard. "
        "Forecast-based planners could use corrected forecasts to rank schedules and used the same simulator for scoring. "
        "Simulated updates supplied reference SOH and delivered Wh. The last measured physical discharge curve was held "
        "fixed during simulated rollouts. The original study and the 25% larger disturbance study are shown below.",
        "",
        "| Disturbance | Policy | Mean SOH loss (pp) | Worst 10% loss (pp) | Maximum SOH loss (pp) | Budget hit | Support stops |",
        "| --- | --- | ---: | ---: | ---: | ---: | ---: |"])
    for row in controller_metrics.itertuples(index=False):
        lines.append(f"| {row.evaluation.title()} | {POLICIES[row.policy]} | {row.soh_loss_pp:.3f} | {row.tail_loss_pp:.3f} | "
                     f"{row.max_loss_pp:.3f} | {row.budget_hit:.2%} | {row.unsupported_stop:.2%} |")
    lines.extend(["", "### Paired comparisons against original Lookahead", "",
        "| Policy | Mean loss reduction | Worst 10% loss reduction | Mean change 95% interval (pp) | Qualifies |",
        "| --- | ---: | ---: | ---: | --- |"])
    for row in controller_comparisons.loc[controller_comparisons.evaluation.eq("original")].itertuples(index=False):
        qualifies = "Yes" if row.qualifies else "No"
        lines.append(f"| {POLICIES[row.policy]} | {row.mean_loss_reduction_pct:+.1f}% | {row.tail_loss_reduction_pct:+.1f}% | "
                     f"{row.mean_ci95_low_pp:+.3f} to {row.mean_ci95_high_pp:+.3f} | {qualifies} |")
    lines.extend(["", "Advancement requires a reduction of at least 10% in mean or worst-decile SOH loss, a negative "
                  "upper paired 95% confidence bound for that metric, no increase in the other SOH loss metrics "
                  "or support stops, and a budget hit rate decline of at most one percentage point.", "",
                  "### Discrete SAC by seed", "",
                  "| Seed | Mean SOH loss (pp) | Worst 10% loss (pp) | Maximum loss (pp) | Budget hit |",
                  "| ---: | ---: | ---: | ---: | ---: |"])
    for row in controller_seed_metrics.loc[controller_seed_metrics.evaluation.eq("original") &
                                          controller_seed_metrics.policy.eq("sac")].itertuples(index=False):
        lines.append(f"| {row.seed} | {row.soh_loss_pp:.3f} | {row.tail_loss_pp:.3f} | "
                     f"{row.max_loss_pp:.3f} | {row.budget_hit:.2%} |")
    lines.extend(["", "SAC uses the same simulated training transitions as FQI. The finite-horizon reward is the "
                  "reference SOH change, with a terminal penalty of three per Wh of energy shortfall and a "
                  "five-point penalty for stopping outside support. Controller intervals resample batteries.", "",
                  "### Planning headroom in the simulator", "",
                  "| Disturbance | Episodes with a feasible complete schedule | Lookahead matches hindsight minimum |",
                  "| --- | ---: | ---: |"])
    for mode, values in oracle_summary().iterrows():
        lines.append(f"| {mode.title()} | {values.attainable} | {values.equal} |")
    lines.extend(["", "The hindsight calculation enumerates complete, guard-compliant schedules that meet the "
                  "original energy budget. It has access to the full simulator disturbance trajectory for each "
                  "episode. This bound applies to the fixed set of templates and to the transition assumptions "
                  "in this simulator.", "",
                  "[XJTU protocols](https://wang-fujin.github.io/) · "
                  "[XJTU dataset](https://zenodo.org/records/10963339) · "
                  "[Discrete SAC](https://arxiv.org/abs/1910.07207)", ""])
    return "\n".join(lines)


controller_metrics, controller_comparisons, selected_policy = controller_scores()
controller_seed_metrics = per_seed_scores()
fixed_decisions = forecast_decisions(forecast_metrics, "horizon_cycles", "selected")
rolling_decisions = forecast_decisions(rolling_metrics, "elapsed_cycles", "bias_corrected")

output.mkdir(parents=True, exist_ok=True)
tables = dict(forecast_predictions=forecast_predictions, forecast_metrics=forecast_metrics, forecast_selection=forecast_selection,
              forecast_decisions=fixed_decisions, rolling_predictions=rolling_predictions, rolling_metrics=rolling_metrics,
              rolling_decisions=rolling_decisions, rolling_selection=rolling_selection, forecast_health_bands=forecast_cases(),
              forecast_peer_sensitivity=peer_sensitivity, reference_curve_features=rpt_curves,
              controller_episodes=controller_episodes, controller_cases=controller_cases, controller_metrics=controller_metrics,
              controller_comparisons=controller_comparisons, controller_seed_metrics=controller_seed_metrics,
              sac_training=sac_training, oracle_headroom=oracle_cases)
tables.update(refinement_predictions=refinement_predictions, refinement_metrics=refinement_metrics,
              refinement_selections=refinement_selections, refinement_health_bands=refinement_health_bands,
              refinement_difficult_cases=refinement_difficult_cases)
for name, table in tables.items():
    destination = output / f"{name}.csv"
    if controller_archive is not None and name in CONTROLLER_CACHED_TABLES:
        saved = artifact(destination.name, controller_archive)
        if not destination.is_file() or destination.read_bytes() != saved:
            destination.write_bytes(saved)
    else:
        table.to_csv(destination, index=False)
configuration = dict(forecast_cycles=[h*BLOCK for h in HORIZONS], forecast_metric_weights=FORECAST_WEIGHTS.tolist(),
    forecast_goal=0.50, controller_selected=selected_policy, operating_cycles_per_action=BLOCK,
    starts=133, disturbance_seed=SEED, disturbance_paths=DISTURBANCES, stress_scale=STRESS_SCALE,
    conservative_planning_paths=PLANNING_PATHS, planning_energy_probability=PLANNING_RISK,
    energy_budget_fraction=ENERGY_FRACTION, qualification_loss_reduction=ADVANCEMENT,
    energy_budget_hit_tolerance=ENERGY_RATE_TOLERANCE, seeds=SEEDS, sac_updates=SAC_UPDATES, sac_batch=SAC_BATCH,
    gru_epochs=GRU_EPOCHS, threads=1, controller_reused=controller_archive is not None,
    forecast_revision=dict(blend_weights=BLEND_WEIGHTS, soh_caps=SOH_CAPS, wh_caps_per_block=WH_CAPS_PER_BLOCK))
(output / "run_config.json").write_text(json.dumps(configuration, indent=2) + "\n", encoding="utf-8")
(output / "summary.md").write_text(describe(), encoding="utf-8")
actor_path = output / "sac_actor_weights.pt"
if controller_archive is None:
    torch.save(actor_weights, actor_path)
else:
    saved_actors = artifact(actor_path.name, controller_archive)
    if not actor_path.is_file() or actor_path.read_bytes() != saved_actors:
        actor_path.write_bytes(saved_actors)
names = [*(f"{name}.csv" for name in tables), "run_config.json", "summary.md", "sac_actor_weights.pt"]
with ZipFile(output.with_suffix(".zip"), "w", ZIP_DEFLATED) as archive:
    for name in names:
        archive.write(output / name, arcname=f"{output.name}/{name}")
print(controller_metrics.loc[controller_metrics.evaluation.eq("original")].round(4).to_string(index=False))
print(f"Selected controller is {POLICIES[selected_policy]}.")
print(f"Report saved to {output / 'summary.md'}")

evaluation    policy  soh_loss_pp  delivered_wh  budget_hit  unsupported_stop  tail_loss_pp  max_loss_pp  max_shortfall_wh  episodes  selected
  original     fixed       1.9844      189.7686      0.9961            0.0039        7.2685      10.4796           40.0674      1064     False
  original       fqi       1.8996      189.8984      0.9961            0.0039        7.2888      10.4830           40.2127      1064     False
  original lookahead       1.7547      190.2481      0.9961            0.0039        7.1017      10.4796           40.0449      1064      True
  original       mpc       1.8420      190.0863      0.9961            0.0039        7.1572      10.5212           40.0449      1064     False
  original       ppo       1.9853      189.7886      0.9961            0.0039        7.3645      10.6980           40.2127      3192     False
  original      risk       1.8421      190.0863      0.9961            0.0039        7.1582      10.5212           40.0449      1064     False